# Atmospheric-model sensitivity in ground-to-air FSO receiver design — v4.1
**Nishat Yasmin Anika · ECE · KUET**  
Standalone CPU notebook generated from canonical source files. Run all cells in order; no Drive mount required.

**Scope:** conditional optical-power availability, not BER, measured uptime or practical receiver specifications. Literal dB/km is the primary convention; equation-as-printed is a legacy comparison only. Table II, the clear-weather prose alternative and the hypothetical radiation-fog stress case are kept separate. No source MODTRAN data or field observations were reproduced. See the embedded source map.

The original repository is [ground-to-air-fso-link-design](https://github.com/Nishat-Yasmin-Anika/ground-to-air-fso-link-design). AI tools assisted code revision and review; author approval is required before publication.

## 1. Mathematical model and verification
For integrated power attenuation $Q$ in dB, $T=10^{-Q/10}$. If the printed quantity instead represents Napierian optical depth, $T=e^{-Q}$. A circular aperture captures $G=F_{\chi'^2_2}((2a/w)^2;\lambda=(2r/w)^2)$ from a displaced Gaussian spot. Independent equal-variance transverse Gaussian jitter implies Rayleigh radial displacement. Received power is $P=P_{tx}\eta T G$; availability is $\Pr(P\ge P_{min})$. TV robustness minimizes the scenario-weighted success probability over a half-L1 probability ball. Feasible-endpoint bisection finds the minimum radius in the specified search interval, including threshold steps at zero jitter. All inputs are conditional assumptions. See the embedded technical note for derivations and limitations.

In [ ]:
import sys, subprocess, importlib.util
from pathlib import Path
missing = [p for p in ('numpy','scipy','matplotlib') if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import tempfile
PROJECT_ROOT = Path(tempfile.mkdtemp(prefix='fso_v41_project_', dir=Path.cwd()))
print('Project:', PROJECT_ROOT)

## 2. Canonical source files
These cells are generated from the repository. To edit later, change the standalone source files and rebuild this notebook rather than maintaining two copies.

### `src/atmosphere.py`

In [ ]:
p = PROJECT_ROOT / 'src/atmosphere.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Transparent 1550-nm atmospheric-only FSO simulation.\n\nNumerical coefficients: Erdogan et al. (2026), IEEE Vehicular Technology\nMagazine, Table I/II. Paper: https://doi.org/10.1109/MVT.2026.3687349\n\nImportant unit ambiguity: Table I reports dB/km but Eqs. (1)-(3) insert\ncoefficients directly into exp(-sum coefficient * path). We therefore support\nT = exp(-Q) (\'paper_exponential\') and T = 10**(-Q/10) (\'db_label\').\nNeither interpretation is claimed to be independently physically validated.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom itertools import product\nfrom math import cos, exp, isclose, isfinite, log, radians, sqrt\nimport numpy as np\n\nPAPER_EXPONENTIAL = \'paper_exponential\'\nDB_LABEL = \'db_label\'\nCONVENTIONS = (PAPER_EXPONENTIAL, DB_LABEL)\nWAVELENGTH_NM = 1550\nMAX_ZENITH_DEG = 70.0\nPROFILES = (\'table_ii\', \'clear_prose\', \'fog_visibility_stress\')\n\n\n@dataclass(frozen=True)\nclass State:\n    name: str\n    probability: float\n    printed_coefficient_per_km: float\n\n    def __post_init__(self):\n        if not (isfinite(self.probability) and 0 <= self.probability <= 1):\n            raise ValueError(\'State probability must be finite and in [0,1]\')\n        if not (isfinite(self.printed_coefficient_per_km)\n                and self.printed_coefficient_per_km >= 0):\n            raise ValueError(\'Coefficient must be nonnegative and finite\')\n\n\n@dataclass(frozen=True)\nclass Layer:\n    name: str\n    bottom_km: float\n    top_km: float\n    states: tuple[State, ...]\n\n    def __post_init__(self):\n        if not (isfinite(self.bottom_km) and isfinite(self.top_km)\n                and 0 <= self.bottom_km < self.top_km):\n            raise ValueError(\'Layer boundaries must be ordered, finite, nonnegative\')\n        if not self.states or not isclose(sum(s.probability for s in self.states), 1.0,\n                                          rel_tol=0, abs_tol=1e-11):\n            raise ValueError(\'Mutually exclusive layer-state probabilities must sum to 1\')\n\n\n@dataclass(frozen=True)\nclass Scenario:\n    name: str\n    layers: tuple[Layer, ...]\n\n    def __post_init__(self):\n        if not self.layers or self.layers[0].bottom_km != 0:\n            raise ValueError(\'Scenarios must start at ground level\')\n        for left, right in zip(self.layers[:-1], self.layers[1:]):\n            if not isclose(left.top_km, right.bottom_km, abs_tol=1e-11):\n                raise ValueError(\'Layers must be contiguous without overlaps or gaps\')\n\n    @property\n    def top_km(self) -> float:\n        return self.layers[-1].top_km\n\n\ndef _slab(name: str, lower: float, upper: float,\n          *states: tuple[str, float, float]) -> Layer:\n    return Layer(name, lower, upper, tuple(State(*x) for x in states))\n\n\ndef scenarios_1550nm(profile: str = "table_ii") -> dict[str, Scenario]:\n    """Transcription of Table II, with rainy clear-air 14.2 km split around cloud.\n\n    Table II prints coefficients as dB/km, but Eqs. 1-3 use exp(-coefficient*km).\n    The five *conditional* scenarios are not assigned probabilities relative\n    to each other. Within a slab, probabilities apply to mutually exclusive states.\n    """\n    clear = (\'clear air\', 1.0, 0.0025)\n    volcanic = _slab(\'stratospheric volcanic aerosol\', 15, 30,\n                     (\'high volcanic activity\', .5, .0104),\n                     (\'background volcanic activity\', .5, .0002036))\n    if profile not in PROFILES:\n        raise ValueError(\'Unknown atmospheric profile\')\n    scenarios = {\n        \'rain\': Scenario(\'Rain / nimbostratus\', (\n            _slab(\'lower clear air\', 0, .6, clear),\n            _slab(\'nimbostratus cloud\', .6, 1.4,\n                  (\'nimbostratus\', .9, 8.2425), (\'normal air\', .1, .034)),\n            _slab(\'upper clear air\', 1.4, 15, clear), volcanic)),\n        \'fog\': Scenario(\'Two-layer fog\', (\n            _slab(\'thick fog\', 0, 1, (\'thick fog\', 1, 1.7680)),\n            _slab(\'light fog\', 1, 2, (\'light fog\', 1, .4592)),\n            _slab(\'upper clear air\', 2, 15, clear), volcanic)),\n        \'clear\': Scenario(\'Clear atmosphere\', (\n            _slab(\'normal lower air\', 0, 3, (\'normal air\', 1, .034)),\n            _slab(\'upper clear air\', 3, 15, clear), volcanic)),\n        \'pollution\': Scenario(\'Extreme air pollution\', (\n            _slab(\'mixed urban aerosol\', 0, 3,\n                  (\'extremely polluted\', .7, .3536), (\'normal air\', .3, .034)),\n            _slab(\'upper clear air\', 3, 15, clear), volcanic)),\n        \'snow\': Scenario(\'Heavy snow\', (\n            _slab(\'heavy snow\', 0, 2, (\'heavy snow\', 1, .2)),\n            _slab(\'upper clear air\', 2, 15, clear), volcanic)),\n    }\n    # Preserve the published table; alternative profiles are explicit, not corrections.\n    if profile == \'clear_prose\':\n        scenarios[\'clear\'] = Scenario(\'Clear atmosphere: Section IV-C prose\', (\n            _slab(\'clear air\', 0, 15, clear), volcanic))\n    elif profile == \'fog_visibility_stress\':\n        # Al Naboulsi radiation-fog polynomial in native km^-1, converted to dB/km.\n        # See doi:10.3390/photonics13111013, Table 1 (1550 nm, V=0.05–1 km).\n        # Only fog coefficients change; layer thicknesses remain Table II.\n        scenarios[\'fog\'] = Scenario(\'Radiation-fog coefficient stress case\', (\n            _slab(\'thick fog\', 0, 1, (\'thick fog\', 1, radiation_fog_db_per_km(.20))),\n            _slab(\'light fog\', 1, 2, (\'light fog\', 1, radiation_fog_db_per_km(.77))),\n            _slab(\'upper clear air\', 2, 15, clear), volcanic))\n    return scenarios\n\n\ndef radiation_fog_db_per_km(visibility_km: float, wavelength_um: float = 1.55) -> float:\n    """Empirical radiation-fog sensitivity model, not local calibration.\n\n    Wavelength in micrometres, visibility in km, output in dB/km.\n    Native extinction and conversion as in doi:10.3390/photonics13111013, Table 1.\n    """\n    if not (.05 <= visibility_km <= 1 and .69 <= wavelength_um <= 1.55):\n        raise ValueError(\'Outside stated radiation-fog model domain\')\n    return (10/log(10))*(.18126*wavelength_um**2 + .13709*wavelength_um + 3.7502)/visibility_km\n\n\ndef path_lengths_km(scenario: Scenario, altitude_km: float,\n                    zenith_deg: float = 0) -> np.ndarray:\n    """Flat, horizontal slab intersection. Zenith = angle from local vertical."""\n    if not (isfinite(altitude_km) and 0 <= altitude_km <= scenario.top_km):\n        raise ValueError(f\'Altitude must be 0 to {scenario.top_km} km\')\n    if not (isfinite(zenith_deg) and 0 <= zenith_deg <= MAX_ZENITH_DEG):\n        raise ValueError(f\'Zenith must be 0 to {MAX_ZENITH_DEG} degrees\')\n    secant = 1.0 / cos(radians(zenith_deg))\n    return np.array([max(0., min(altitude_km, l.top_km) - l.bottom_km) * secant\n                     for l in scenario.layers], dtype=float)\n\n\ndef integrated_printed_coefficient(scenario: Scenario, altitude_km: float,\n                                   zenith_deg: float = 0) -> float:\n    """Q = sum(d_j*sum_k(p_jk*c_jk)); dimension depends on interpretation."""\n    ds = path_lengths_km(scenario, altitude_km, zenith_deg)\n    return float(sum(d*sum(s.probability*s.printed_coefficient_per_km for s in layer.states)\n                     for d, layer in zip(ds, scenario.layers)))\n\n\ndef transmittance_from_q(q: float | np.ndarray,\n                         convention: str = DB_LABEL):\n    """Two explicitly separate readings of the source\'s coefficient notation."""\n    if convention not in CONVENTIONS:\n        raise ValueError(f\'Unknown convention {convention!r}\')\n    q = np.asarray(q, dtype=float)\n    if np.any(~np.isfinite(q)) or np.any(q < 0):\n        raise ValueError(\'Q must be finite and nonnegative\')\n    val = np.exp(-q) if convention == PAPER_EXPONENTIAL else np.power(10., -q/10.)\n    return float(val) if val.ndim == 0 else val\n\n\ndef loss_db_from_q(q: float | np.ndarray,\n                   convention: str = DB_LABEL):\n    """Equivalent power loss in dB, *after* choosing a convention."""\n    if convention not in CONVENTIONS:\n        raise ValueError(\'Unknown convention\')\n    q = np.asarray(q, dtype=float)\n\n    if np.any(~np.isfinite(q)) or np.any(q < 0):\n        raise ValueError(\'Q must be finite and nonnegative\')\n\n    return q * (10. / log(10.) if convention == PAPER_EXPONENTIAL else 1.)\n\n\ndef effective_transmittance(scenario: Scenario, altitude_km: float,\n                            zenith_deg: float = 0,\n                            convention: str = DB_LABEL) -> float:\n    """T_eff = f(E[Q]); *not* generally equal to E[f(Q)]."""\n    return transmittance_from_q(integrated_printed_coefficient(\n        scenario, altitude_km, zenith_deg), convention)\n\n\ndef exact_distribution(scenario: Scenario, altitude_km: float,\n                       zenith_deg: float = 0) -> tuple[np.ndarray, np.ndarray]:\n    """Enumerate independent per-layer categorical states (at most 4 here).\n\n    Returns Q values and their probabilities. Independence is an explicit\n    teaching assumption, NOT a weather-correlation estimate from ERA5.\n    Complexity O(N_layers * product_j K_j); suitable ONLY for few states.\n    """\n    ds = path_lengths_km(scenario, altitude_km, zenith_deg)\n    active = [(l, d) for l, d in zip(scenario.layers, ds) if d > 0]\n    if not active:\n        return np.array([0.]), np.array([1.])\n    q_values, probabilities = [], []\n    for selected in product(*(layer.states for layer, _ in active)):\n        q_values.append(sum(state.printed_coefficient_per_km*d\n                            for state, (_, d) in zip(selected, active)))\n        probabilities.append(float(np.prod([s.probability for s in selected])))\n    probs = np.asarray(probabilities, dtype=float)\n    if not np.isclose(probs.sum(), 1.0, atol=1e-12):\n        raise RuntimeError(\'Enumeration probabilities do not normalize\')\n    return np.asarray(q_values, dtype=float), probs\n\n\ndef exact_statistics(scenario: Scenario, altitude_km: float,\n                     zenith_deg: float = 0, budget_db: float = 3.,\n                     convention: str = DB_LABEL) -> dict[str, float]:\n    """Exact finite-state E[T] and atmospheric loss-budget exceedance.\n\n    Budget exceedance is NOT receiver outage: receiver, scintillation,\n    pointing loss, geometric spread, photodetector noise are absent.\n    """\n    if not (isfinite(budget_db) and budget_db >= 0):\n        raise ValueError(\'Budget must be finite and nonnegative\')\n    q, p = exact_distribution(scenario, altitude_km, zenith_deg)\n    t = transmittance_from_q(q, convention)\n    return {\'mean_q\': float(np.dot(p, q)),\n            \'effective_transmittance\': effective_transmittance(\n                scenario, altitude_km, zenith_deg, convention),\n            \'mean_transmittance\': float(np.dot(p, t)),\n            \'exceedance_probability\': float(np.dot(p, loss_db_from_q(q, convention) > budget_db)),\n            \'number_of_combinations\': len(q)}\n\n\ndef mean_transmittance_factorized(scenario: Scenario, altitude_km: float,\n                                  zenith_deg: float = 0,\n                                  convention: str = DB_LABEL) -> float:\n    """O(sum K_j) analytic E[T] if states in different slabs are independent."""\n    ds = path_lengths_km(scenario, altitude_km, zenith_deg)\n    product_mean = 1.\n    for l, d in zip(scenario.layers, ds):\n        product_mean *= sum(s.probability * transmittance_from_q(\n            s.printed_coefficient_per_km*d, convention) for s in l.states)\n    return float(product_mean)\n\n\ndef sample_q(scenario: Scenario, altitude_km: float, zenith_deg: float = 0,\n             n: int = 10000, seed: int = 20261123) -> np.ndarray:\n    """Vectorized categorical Monte Carlo (MC), only for cross-checking."""\n    if not isinstance(n, (int, np.integer)) or n < 1:\n        raise ValueError(\'Sample count must be positive integer\')\n    ds = path_lengths_km(scenario, altitude_km, zenith_deg)\n    rng = np.random.default_rng(seed)  # PCG64 as currently selected by NumPy default_rng\n    total = np.zeros(n, dtype=float)\n    for l, d in zip(scenario.layers, ds):\n        if d == 0:\n            continue\n        coeffs = np.array([s.printed_coefficient_per_km for s in l.states])\n        probs = np.array([s.probability for s in l.states])\n        total += rng.choice(coeffs, size=n, p=probs) * d\n    return total\n\n\ndef wilson_interval(successes: int, n: int, z: float = 1.959963984540054):\n    """Wilson score interval for Bernoulli trials, approximately 95% for z=1.96."""\n    if (\n        not isinstance(n, (int, np.integer))\n        or not isinstance(successes, (int, np.integer))\n        or isinstance(n, (bool, np.bool_))\n        or isinstance(successes, (bool, np.bool_))\n        or n < 1\n        or not 0 <= successes <= n\n    ):\n        raise ValueError(\'Invalid count\')\n\n    if not isfinite(z) or z <= 0:\n        raise ValueError(\'Confidence multiplier must be finite and positive\')\n    p = successes / n\n    denom = 1 + z*z/n\n    center = (p + z*z/(2*n))/denom\n    half = z * sqrt(p*(1-p)/n + z*z/(4*n*n))/denom\n    return max(0., center-half), min(1., center+half)\n\n\ndef uniform_ground_baseline_q(scenario: Scenario, altitude_km: float,\n                              zenith_deg: float = 0) -> float:\n    """Deliberately naive surface-only baseline; NOT another published model."""\n    path_lengths_km(scenario, altitude_km, zenith_deg)\n    bottom = scenario.layers[0]\n    ground_coefficient = sum(s.probability*s.printed_coefficient_per_km\n                             for s in bottom.states)\n    return ground_coefficient*altitude_km/cos(radians(zenith_deg))\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `src/fso_link.py`

In [ ]:
p = PROJECT_ROOT / 'src/fso_link.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Research-grade *illustrative* ground-to-air FSO link design study.\n\nModel components:\n* atmosphere.py transcribes Erdogan et al. (2026) Tables I-II with dual-unit audit\n* circular-aperture capture fraction for an offset isotropic 2D Gaussian beam\n* independent, zero-mean Gaussian horizontal/vertical pointing-jitter components\n* conditional exact optical-power availability (weather mixture, Rayleigh jitter)\n* worst-case probability under total-variation (TV) distance <= rho from a nominal\n  **illustrative**, not empirically estimated, weather-scenario mixture.\n\nThis is not a complete end-to-end communications BER/outage model: atmospheric\nscintillation, turbulence fading, pointing temporal dynamics, noise and detector\nthermal effects are excluded, and sensitivity is a demonstration threshold.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass, asdict\nimport math\nfrom typing import Dict\nimport numpy as np\nfrom scipy.stats import ncx2\nfrom scipy.optimize import brentq, linprog\nfrom scipy.integrate import quad\nfrom scipy.special import i0e\nfrom atmosphere import (PAPER_EXPONENTIAL, DB_LABEL, scenarios_1550nm,\n                        exact_distribution, transmittance_from_q, PROFILES)\n\nSCENARIO_NAMES = (\'clear\',\'pollution\',\'fog\',\'rain\',\'snow\')\n# Hypothetical demonstration mixture, NOT ERA5-derived; source paper does\n# NOT provide a probability distribution across the five weather scenarios.\nDEFAULT_PRIOR = np.array([.76, .10, .07, .04, .03])\n\n@dataclass(frozen=True)\nclass Link:\n    altitude_km: float = 3.0\n    zenith_deg: float = 0.0\n    tx_mw: float = 50.0\n    optical_efficiency: float = 0.70\n    threshold_mw: float = 2.0\n    beam_waist_m: float = .025\n    divergence_urad: float = 55.0\n    convention: str = DB_LABEL\n    profile: str = "table_ii"\n\n    def __post_init__(self):\n        for name in (\'altitude_km\',\'tx_mw\',\'optical_efficiency\',\'threshold_mw\',\'beam_waist_m\',\'divergence_urad\'):\n            x = getattr(self, name)\n            if not np.isfinite(x) or x <= 0:\n                raise ValueError(f\'{name} must be positive and finite\')\n        if not (0 < self.optical_efficiency <= 1):\n            raise ValueError(\'optical efficiency outside (0,1]\')\n        if not (0 <= self.zenith_deg <= 70):\n            raise ValueError(\'Zenith out of range\')\n        if self.convention not in (PAPER_EXPONENTIAL,DB_LABEL):\n            raise ValueError(\'Unknown atmospheric convention\')\n        if self.profile not in PROFILES:\n            raise ValueError(\'Unknown atmospheric profile\')\n        if self.profile == \'fog_visibility_stress\' and self.convention != DB_LABEL:\n            raise ValueError(\'Radiation-fog model explicitly uses dB/km\')\n        if self.implied_m_squared < 1 - 1e-12:\n            raise ValueError(\'Waist/divergence imply M² below diffraction limit\')\n        if self.altitude_km > 30:\n            raise ValueError(\'Source scenarios terminate at 30 km\')\n\n    @property\n    def length_m(self):\n        # Flat Earth and parallel horizontal layers as in atmospheric slab model\n        return 1000 * self.altitude_km / math.cos(math.radians(self.zenith_deg))\n\n    @property\n    def implied_m_squared(self):\n        """Gaussian-equivalent beam quality; not a measured M²."""\n        return math.pi*self.beam_waist_m*self.divergence_urad*1e-6/1550e-9\n\n    @property\n    def beam_radius_m(self):\n        # Simplified far-field Gaussian 1/e^2 intensity radius; **not** a\n        # diffraction-limited propagation calculation, beam waist/divergence\n        # are independent illustrative inputs.\n        return math.hypot(self.beam_waist_m, self.divergence_urad*1e-6*self.length_m)\n\n\ndef gaussian_aperture_capture(a_m: float, w_m: float, displacement_m):\n    """Exact integral of 2-D normalized Gaussian intensity over circular aperture.\n\n    I(x,y) = 2/(pi*w^2) exp(-2*||[x,y]-[r,0]||^2/w^2).\n    Integrated power = F_{noncentral chi-square(2, lambda)}((2a/w)^2),\n    lambda=(2r/w)^2. This is geometrical+pointing collection only.\n    """\n    if not (math.isfinite(a_m) and math.isfinite(w_m) and a_m > 0 and w_m > 0):\n        raise ValueError(\'Aperture radius and beam radius must be positive\')\n    r = np.asarray(displacement_m)\n    if np.any(~np.isfinite(r)) or np.any(r < 0):\n        raise ValueError(\'Displacement must be >=0\')\n    x = (2*a_m/w_m)**2\n    return ncx2.cdf(x, df=2, nc=(2*r/w_m)**2)\n\n\ndef capture_via_independent_quadrature(a: float,w: float,r: float) -> float:\n    """Independent radial integration using scaled modified Bessel I_0.\n\n    4s/w^2 * exp(-2*(s-r)^2/w^2) * i0e(4sr/w^2).\n    """\n    if not all(math.isfinite(v) for v in (a,w,r)) or a <= 0 or w <= 0 or r < 0:\n        raise ValueError(\'Invalid aperture geometry\')\n    def f(s):\n        z = 4*s*r/(w*w)\n        return 4*s/(w*w) * math.exp(-2*(s-r)**2/(w*w)) * i0e(z)\n    return quad(f,0,a,epsabs=1e-11,epsrel=1e-10)[0]\n\n\ndef mean_capture_analytic(a: float,w: float,sigma_m: float):\n    """Exact E[g(R)] for zero-mean 2D Gaussian jitter with axis SD sigma_m.\n\n    Follows by convolution of independent 2D Gaussian photon location\n    N(0,w^2/4 I) and pointing offset N(0,sigma^2 I).\n    """\n    if not all(math.isfinite(v) for v in (a,w,sigma_m)) or a <= 0 or w <= 0 or sigma_m < 0:\n        raise ValueError(\'Invalid aperture geometry or jitter\')\n    return -math.expm1(-a*a/(2*(sigma_m*sigma_m+w*w/4)))\n\n\ndef critical_offset(a: float,w: float,required_capture: float) -> float:\n    """Find largest radial displacement allowing capture >= required_capture.\n\n    Monotonicity is valid for a centered circular receiver and a radially\n    symmetric Gaussian beam (no boresight offset). Returns 0 if impossible.\n    """\n    if not all(math.isfinite(v) for v in (a,w,required_capture)) or a <= 0 or w <= 0:\n        raise ValueError("Invalid aperture or required capture")\n    if required_capture <= 0: return math.inf\n    g0 = -math.expm1(-2*a*a/(w*w))\n    if required_capture >= g0: return 0.0\n    # Gaussian tail bound: if r>a, capture <= exp[-2(r-a)^2/w²].\n    hi = a + w*math.sqrt(-math.log(required_capture)/2) + w\n    if not math.isfinite(hi): raise ValueError(\'Geometry exceeds numeric range\')\n    return brentq(lambda r: gaussian_aperture_capture(a,w,r)-required_capture,\n                  0,hi, xtol=1e-12, rtol=1e-11)\n\n\ndef success_probability_for_transmittance(t: float,a: float,jitter_urad:float,link:Link) -> float:\n    """Exact P(P_rx >= P_threshold) conditioned on atmospheric state.\n\n    Jitter is independent zero-mean Gaussian in x/y (Rayleigh magnitude).\n    """\n    if not math.isfinite(jitter_urad) or jitter_urad < 0: raise ValueError(\'Invalid jitter\')\n    if not math.isfinite(a) or a <= 0: raise ValueError(\'Invalid aperture\')\n    if not math.isfinite(t) or not 0 <= t <= 1: raise ValueError(\'Invalid transmittance\')\n    if t == 0: return 0.0\n    needed = link.threshold_mw/(link.tx_mw*link.optical_efficiency*t)\n    if needed >= 1: return 0.0\n    rcrit = critical_offset(a,link.beam_radius_m,needed)\n    if math.isinf(rcrit): return 1.\n    if jitter_urad == 0:\n        return float(needed <= -math.expm1(-2*a*a/link.beam_radius_m**2))\n    sigma_m = jitter_urad*1e-6*link.length_m\n    return -math.expm1(-0.5*(rcrit/sigma_m)**2)\n\n\ndef conditional_availability(a:float,jitter_urad:float,link:Link) -> np.ndarray:\n    """Exact within-scenario categorical-state mixture + Rayleigh jitter CDF."""\n    if a <= 0: raise ValueError(\'a must be positive\')\n    values=[]\n    for name in SCENARIO_NAMES:\n        q,prob=exact_distribution(scenarios_1550nm(link.profile)[name],link.altitude_km,link.zenith_deg)\n        trans=transmittance_from_q(q,link.convention)\n        chance=[success_probability_for_transmittance(float(t),a,jitter_urad,link) for t in trans]\n        values.append(float(np.clip(np.dot(prob,chance),0,1)))\n    return np.array(values)\n\n\ndef validate_prior(q):\n    q=np.asarray(q,dtype=float)\n    if q.shape != (len(SCENARIO_NAMES),) or np.any(q<0) or not np.all(np.isfinite(q)) or abs(q.sum()-1)>1e-10:\n        raise ValueError(\'Prior must be 5 nonnegative normalized probabilities\')\n    return q\n\n\ndef worst_case_tv_greedy(chances, prior, rho:float) -> float:\n    """Minimize p·chances for p in simplex and total-variation(p,prior)<=rho.\n\n    For linear objectives, transfer probability mass from high-reward states\n    to low-reward states until the TV budget is used. Complexity O(S log S).\n    """\n    p=validate_prior(prior).copy()\n    a=np.asarray(chances,dtype=float)\n    if a.shape!=p.shape or np.any(~np.isfinite(a)) or np.any(a<0) or np.any(a>1):\n        raise ValueError(\'Invalid availability vector\')\n    if not 0<=rho<=1: raise ValueError(\'TV radius must be in [0,1]\')\n    ranks=np.argsort(a)\n    lo, hi=0,len(a)-1\n    budget=rho\n    while lo<hi and budget>1e-14:\n        donor=int(ranks[hi]); recipient=int(ranks[lo])\n        if a[donor]-a[recipient]<=1e-13: break\n        mass=min(budget,p[donor],1-p[recipient])\n        if mass <= 1e-14:\n            if p[donor]<=1e-14: hi-=1\n            if 1-p[recipient]<=1e-14: lo+=1\n            continue\n        p[donor]-=mass; p[recipient]+=mass; budget-=mass\n        if p[donor]<=1e-14: hi-=1\n        if 1-p[recipient]<=1e-14: lo+=1\n    return float(np.clip(np.dot(p,a),0,1))\n\n\ndef worst_case_tv_lp(chances, prior, rho:float):\n    """Independent linear-programming reference for the greedy TV solver."""\n    p0=validate_prior(prior)\n    a=np.asarray(chances,dtype=float)\n    if a.shape!=p0.shape or np.any(~np.isfinite(a)) or np.any(a<0) or np.any(a>1) or not 0<=rho<=1:\n        raise ValueError(\'Invalid LP inputs\')\n    n=len(p0)\n    cost=np.r_[a,np.zeros(n)]\n    A=[]; b=[]\n    for i in range(n):\n        row=np.zeros(2*n);row[i]=1;row[n+i]=-1; A.append(row);b.append(p0[i])\n        row=np.zeros(2*n);row[i]=-1;row[n+i]=-1; A.append(row);b.append(-p0[i])\n    row=np.r_[np.zeros(n),np.ones(n)];A.append(row);b.append(2*rho)\n    eq=np.r_[np.ones(n),np.zeros(n)][None,:]\n    solution=linprog(cost,A_ub=np.asarray(A),b_ub=np.asarray(b),A_eq=eq,b_eq=[1],bounds=[(0,None)]*(2*n),method=\'highs\')\n    if not solution.success: raise RuntimeError(solution.message)\n    return float(solution.fun)\n\n\ndef availability(a,jitter_urad,link,prior=DEFAULT_PRIOR,rho=0.):\n    vec=conditional_availability(a,jitter_urad,link)\n    if rho==0: return float(np.clip(np.dot(validate_prior(prior),vec),0,1))\n    return worst_case_tv_greedy(vec,prior,rho)\n\n\ndef min_aperture_for_target(target,jitter_urad,link,prior=DEFAULT_PRIOR,rho=0.,a_lo=.025,a_hi=.50,atol=1e-7):\n    """Return the feasible endpoint of a monotone search, within [a_lo,a_hi].\n\n    NaN means no feasible aperture in the search interval; consult the\n    atmosphere-only ceiling separately. Handles discontinuous zero-jitter steps.\n    """\n    if not (0 < target < 1): raise ValueError(\'Invalid target\')\n    if not all(math.isfinite(v) for v in (a_lo,a_hi,atol)) or not 0<a_lo<=a_hi or atol<=0:\n        raise ValueError(\'Invalid aperture interval or tolerance\')\n    f=lambda a: availability(a,jitter_urad,link,prior,rho)\n    if f(a_hi)<target: return math.nan\n    if f(a_lo)>=target: return a_lo\n    lo,hi=a_lo,a_hi\n    while hi-lo>atol:\n        mid=(lo+hi)/2\n        if mid==lo or mid==hi: break\n        if f(mid)>=target: hi=mid\n        else: lo=mid\n    return hi\n\n\ndef sample_link(n:int,a:float,jitter_urad:float,link:Link,prior=DEFAULT_PRIOR,seed=20261123):\n    """Independent Monte Carlo: draw weather scenario, then its slab states,\n    then 2D pointing offsets; evaluate physical collection CDF directly.\n    Does NOT sample Bernoulli successes from analytic availability.\n    """\n    if not isinstance(n,(int,np.integer)) or n<2: raise ValueError(\'Need at least two draws\')\n    if not math.isfinite(a) or a<=0 or not math.isfinite(jitter_urad) or jitter_urad<0:\n        raise ValueError(\'Invalid aperture or jitter\')\n    prior=validate_prior(prior)\n    rng=np.random.default_rng(seed)\n    which=rng.choice(len(SCENARIO_NAMES),size=n,p=prior)\n    q_all=np.empty(n)\n    # Use explicit per-layer independently drawn states, not exact enumerated\n    # results, to provide an independent stochastic implementation.\n    for idx,name in enumerate(SCENARIO_NAMES):\n        mask=which==idx; k=int(mask.sum())\n        if not k:continue\n        scenario=scenarios_1550nm(link.profile)[name]\n        qv=np.zeros(k)\n        for slab in scenario.layers:\n            d=max(0.,min(link.altitude_km,slab.top_km)-slab.bottom_km)/math.cos(math.radians(link.zenith_deg))\n            if d==0: continue\n            coeff=np.array([s.printed_coefficient_per_km for s in slab.states])\n            prob=np.array([s.probability for s in slab.states])\n            qv += d*rng.choice(coeff,size=k,p=prob)\n        q_all[mask]=qv\n    t=transmittance_from_q(q_all,link.convention)\n    sigma=jitter_urad*1e-6*link.length_m\n    x=rng.normal(0,sigma,n);y=rng.normal(0,sigma,n)\n    capture=gaussian_aperture_capture(a,link.beam_radius_m,np.hypot(x,y))\n    power=link.tx_mw*link.optical_efficiency*t*capture\n    passed=(power>=link.threshold_mw) & (link.tx_mw*link.optical_efficiency*t>link.threshold_mw)\n    return {\'n\':int(n),\'success_count\':int(passed.sum()),\'availability\':float(passed.mean()),\n            \'mean_received_mw\':float(power.mean()),\'std_received_mw\':float(power.std(ddof=1))}\n\n\ndef wilson_score(count:int,n:int,z=1.959963984540054):\n    if (\n        not isinstance(n, (int, np.integer))\n        or not isinstance(count, (int, np.integer))\n        or isinstance(n, (bool, np.bool_))\n        or isinstance(count, (bool, np.bool_))\n        or n < 1\n        or count < 0\n        or count > n\n    ):\n        raise ValueError(\'Invalid count\')\n\n    if not math.isfinite(z) or z <= 0:\n        raise ValueError(\'Confidence multiplier must be finite and positive\')\n    phat=count/n; denom=1+z*z/n\n    center=(phat+z*z/(2*n))/denom\n    delta=z/denom*math.sqrt(phat*(1-phat)/n+z*z/(4*n*n))\n    return max(0.,center-delta),min(1.,center+delta)\n\n\ndef weather_limited_ceiling(link:Link, prior=DEFAULT_PRIOR, rho=0.):\n    """Upper bound on availability for ANY finite aperture and jitter.\n\n    In the a -> infinity limit capture -> 1 for each finite offset, so only\n    states with P_tx*eta*T > power threshold can ever pass at finite radius. Hence this\n    is a global model-feasibility certificate, not merely grid infeasibility.\n    """\n    limits=[]\n    for name in SCENARIO_NAMES:\n        q,p=exact_distribution(scenarios_1550nm(link.profile)[name],link.altitude_km,link.zenith_deg)\n        t=transmittance_from_q(q,link.convention)\n        limits.append(float(np.dot(p,link.tx_mw*link.optical_efficiency*t>link.threshold_mw)))\n    conditional=np.array(limits)\n    cap=(float(np.dot(validate_prior(prior),conditional)) if rho==0 else\n         worst_case_tv_greedy(conditional,prior,rho))\n    return cap,conditional\n\n\ndef critical_tv_radius(target:float,link:Link,prior=DEFAULT_PRIOR):\n    """Boundary TV radius where the ceiling reaches target (strict failure beyond).\n\n    This is the exact theoretical break-even radius for the model\'s weather\n    frequency ambiguity, found with a one-dimensional monotone bisection.\n    """\n    if not 0 < target < 1:raise ValueError(\'Invalid target\')\n    f=lambda rho:weather_limited_ceiling(link,prior,rho)[0]-target\n    if f(0)<0:return 0.\n    if f(1)>=0:return math.inf\n    return brentq(f,0,1,xtol=1e-12)\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `run_study.py`

In [ ]:
p = PROJECT_ROOT / 'run_study.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Generate fully reproducible results; launch: python run_study.py.\n\nAll generated artifacts go under results/; no figure is hand-edited.\n"""\nimport os,sys,json,csv,math,time,platform,hashlib, subprocess\nfrom pathlib import Path\nimport numpy as np\nimport scipy\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom dataclasses import asdict, replace\nROOT=Path(__file__).resolve().parent\nsys.path.insert(0,str(ROOT/\'src\'))\nfrom atmosphere import (PAPER_EXPONENTIAL,DB_LABEL,scenarios_1550nm,exact_distribution,\n                        transmittance_from_q, radiation_fog_db_per_km, effective_transmittance, path_lengths_km)\nfrom fso_link import (Link,SCENARIO_NAMES,DEFAULT_PRIOR,gaussian_aperture_capture,\n    capture_via_independent_quadrature,mean_capture_analytic,conditional_availability,\n    availability,worst_case_tv_greedy,worst_case_tv_lp,min_aperture_for_target,\n    sample_link,wilson_score,weather_limited_ceiling,critical_tv_radius, success_probability_for_transmittance)\n\nOUT=ROOT/\'results\'; FIG=OUT/\'figures\'; TBL=OUT/\'tables\'; META=OUT/\'metadata\'\nfor d in (FIG,TBL,META):d.mkdir(parents=True,exist_ok=True)\nlink=Link()  # literal dB/km by default\nSEED=20261123\nTARGET=.92\nRHO=.04\nFAST=os.environ.get(\'FAST_MODE\',\'1\') != \'0\'\nN_MC=20000 if FAST else 100000\nDPI=165\nplt.rcParams.update({\'figure.dpi\':125,\'savefig.dpi\':DPI,\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False})\nstart=time.perf_counter()\n\ndef savefig(name):\n    plt.tight_layout();plt.savefig(FIG/(name+\'.png\'));plt.savefig(FIG/(name+\'.svg\'));plt.close()\n\ndef csvout(name, header, rows):\n    with (TBL/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.writer(f);writer.writerow(header)\n        for row in rows:writer.writerow(row)\n\n# 01: model verification + Gaussian collection geometry\naperture=.14\nr=np.linspace(0,.42,140)\nw=link.beam_radius_m\ng=gaussian_aperture_capture(aperture,w,r)\nind_offsets=[0,.05,.1,.16,.25]\nind_check=np.array([capture_via_independent_quadrature(aperture,w,x) for x in ind_offsets])\nassert np.allclose(ind_check,gaussian_aperture_capture(aperture,w,ind_offsets),atol=1e-10)\nplt.figure(figsize=(7,4.3))\nplt.plot(r,g,label=\'Noncentral chi-square CDF (exact)\')\nplt.scatter(ind_offsets,ind_check,label=\'Independent Bessel quadrature\',marker=\'o\',s=24,zorder=3)\nplt.xlabel(\'Beam-centre displacement at receiver (m)\')\nplt.ylabel(\'Captured optical-power fraction\')\nplt.title(\'01 | Finite circular aperture: exact Gaussian-beam overlap\')\nplt.ylim(-.03,1.03);plt.legend(loc=\'upper right\',fontsize=8)\nsavefig(\'01_aperture_integral_validation\')\n\n# 02: source coefficient convention and different weather paths\nrows=[]\nfor name in SCENARIO_NAMES:\n    q,pr=exact_distribution(scenarios_1550nm()[name],link.altitude_km,link.zenith_deg)\n    for convention in [PAPER_EXPONENTIAL,DB_LABEL]:\n        t=transmittance_from_q(q,convention)\n        rows.append((name,convention,float(np.dot(pr,t)),float(np.min(t)),float(np.max(t)),len(q)))\ncsvout(\'01_weather_scenarios_and_conventions.csv\',[\'scenario\',\'convention\',\'mean_transmittance\',\'minimum_transmittance\',\'maximum_transmittance\',\'finite_states\'],rows)\nplt.figure(figsize=(8.4,4.6));idx=np.arange(len(SCENARIO_NAMES))\ny_a=np.array([r[2] for r in rows[::2]])\ny_b=np.array([r[2] for r in rows[1::2]])\nplt.bar(idx-.19,y_a,width=.38,label=\'Equation-as-printed: exp(-Q)\')\nplt.bar(idx+.19,y_b,width=.38,label=\'Unit-label interpretation: 10^(-Q/10)\')\nplt.xticks(idx,SCENARIO_NAMES)\nplt.ylabel(\'Mean atmospheric transmittance (unitless)\')\nplt.title(\'02 | 3-km atmosphere: published coefficient-notation ambiguity\')\nplt.legend(fontsize=8);savefig(\'02_atmospheric_unit_interpretations\')\n\n# 03: availability vs receiver aperture, at three pointing-jitter levels\nradii=np.linspace(.025,.30,56)\ncalculated={}\nplt.figure(figsize=(8.6,5.1))\nfor jitter in [8,15,25]:\n    nominal=np.array([availability(a,jitter,link) for a in radii]);robust=np.array([availability(a,jitter,link,rho=RHO) for a in radii])\n    calculated[jitter]=(nominal,robust)\n    plt.plot(radii*200,nominal,label=f\'Nominal, jitter {jitter} µrad\')\n    plt.plot(radii*200,robust,linestyle=\'--\',label=f\'TV-robust, jitter {jitter} µrad\')\nplt.axhline(TARGET,color=\'black\',ls=\':\',label=\'Target availability: 92%\')\nplt.xlabel(\'Receiver aperture diameter (cm)\');plt.ylabel(\'Optical-power availability\')\nplt.title(\'03 | Nominal versus uncertain-weather design\')\nplt.ylim(0,1.00);plt.legend(loc=\'lower right\',ncol=2,fontsize=7)\nsavefig(\'03_availability_vs_aperture\')\n\n# 04: design frontier\nfrontier=[]\nfor jitter in [5,8,10,12,15,18,20,22,25,28,30]:\n    for rho in [0,RHO,.08]:\n        a=min_aperture_for_target(TARGET,jitter,link,rho=rho)\n        frontier.append((jitter,rho, float(2*a*100) if np.isfinite(a) else None,\n                         \'feasible\' if np.isfinite(a) else \'infeasible within 1.0m diameter\'))\ncsvout(\'02_minimum_aperture_frontier.csv\',[\'jitter_sigma_per_axis_urad\',\'TV_radius\',\'minimum_aperture_diameter_cm\',\'status\'],frontier)\nplt.figure(figsize=(7.8,4.7))\nfor rho,style in [(0,\'-\'),(RHO,\'--\')]:\n    v=[r for r in frontier if abs(r[1]-rho)<1e-9]\n    y=np.array([np.nan if x[2] is None else x[2] for x in v])\n    plt.plot([x[0] for x in v],y,linestyle=style,marker=\'o\',label=f\'TV radius {rho:.0%}\')\nplt.xlabel(\'Pointing angular jitter: per-axis SD (µrad)\')\nplt.ylabel(\'Minimum receiver aperture diameter (cm)\')\nplt.title(\'04 | Design trade-off: jitter control versus aperture\')\nv=[r for r in frontier if abs(r[1]-.08)<1e-9]\nplt.plot([x[0] for x in v],[np.nan if x[2] is None else x[2] for x in v],linestyle=\':\',marker=\'o\',label=\'TV radius 8%\')\nplt.legend(fontsize=9);savefig(\'04_minimum_aperture_frontier\')\n\n# 05: uncertainty sensitivity at fixed hardware, includes exact LP reference\njitter=15; ap=.08\nvec=conditional_availability(ap,jitter,link)\nrad=np.linspace(0,.16,35)\nvalues=np.array([worst_case_tv_greedy(vec,DEFAULT_PRIOR,float(z)) for z in rad])\nlp_points=[0,.02,.04,.08,.12,.16];lp_res=[worst_case_tv_lp(vec,DEFAULT_PRIOR,z) for z in lp_points]\nassert np.allclose(lp_res,[worst_case_tv_greedy(vec,DEFAULT_PRIOR,z) for z in lp_points],atol=1e-9)\nplt.figure(figsize=(7.7,4.6))\nplt.plot(rad,values,label=\'Exact TV worst-case mass transfer\')\nplt.scatter(lp_points,lp_res,color=\'black\',s=20,label=\'Independent linear program\')\nplt.axhline(TARGET,ls=\':\',color=\'grey\',label=\'92% target\')\nplt.xlabel(\'TV radius: maximum probability mass reassigned\')\nplt.ylabel(\'Worst-case optical-power availability\')\nplt.title(\'05 | Sensitivity to uncertain weather frequencies\')\nplt.ylim(max(0,min(values)-.03),1);plt.legend(fontsize=8);savefig(\'05_weather_probability_shift\')\ncsvout(\'03_tv_radius_sensitivity.csv\',[\'TV_radius\',\'worst_case_availability\'],zip(rad,values))\n\n# 06: feasibility heat map across TV radius and aperture\nrho_grid=np.linspace(0,.10,24)\nap_grid=np.linspace(.025,.20,35)\nmat=np.array([[availability(a,15,link,rho=rr) for a in ap_grid] for rr in rho_grid])\nfig,ax=plt.subplots(figsize=(8.1,5.0))\nim=ax.pcolormesh(ap_grid*200,rho_grid,mat,shading=\'nearest\',cmap=\'viridis\',vmin=float(mat.min()),vmax=1.)\nfig.colorbar(im,ax=ax,label=\'Worst-case optical-power availability\')\ntry: ax.contour(ap_grid*200,rho_grid,mat,levels=[TARGET],colors=\'white\',linewidths=1.6)\nexcept ValueError: pass\nax.set_xlabel(\'Receiver aperture diameter (cm)\');ax.set_ylabel(\'TV radius\')\nax.set_title(\'06 | Feasible region: robust availability threshold at 15 µrad jitter\')\nsavefig(\'06_design_feasibility_map\')\n\n# 07: ablation of weather/jitter vs combined; no false claim of outage BER\nablation=[]\nfor a in radii:\n    joint=availability(a,15,link)\n    weather_only=availability(a,0,link)\n    # No atmospheric extinction; stochastic pointing jitter retained.\n    point_only = success_probability_for_transmittance(1., a, 15, link)\n    ideal=success_probability_for_transmittance(1.,a,0,link)\n    ablation.append((a,joint,weather_only,point_only,ideal))\ncsvout(\'04_ablation.csv\',[\'aperture_radius_m\',\'joint_weather_and_pointing\',\'weather_only_zero_jitter\',\'no_extinction_pointing_only\',\'no_extinction_no_jitter\'],ablation)\nplt.figure(figsize=(8,4.7))\nfor ii,label in [(1,\'Joint: weather + pointing\'),(2,\'Weather mixture; no jitter\'),(3,\'No extinction; jitter\'),(4,\'No extinction; no jitter\')]:\n    plt.plot(radii*200,[row[ii] for row in ablation],label=label)\nplt.axhline(TARGET,ls=\':\',color=\'grey\')\nplt.xlabel(\'Receiver aperture diameter (cm)\');plt.ylabel(\'Nominal optical-power availability\')\nplt.title(\'07 | Ablation: weather state and pointing both matter\')\nplt.ylim(-.01,1.01);plt.legend(fontsize=8);savefig(\'07_ablation\')\n\n# 08: independent Monte Carlo vs exact model and Wilson CIs\nN_levels=[1000,5000,N_MC]\nexact=availability(ap,15,link)\nmc_rows=[]\nfor n in N_levels:\n    st=time.perf_counter(); result=sample_link(n,ap,15,link,seed=SEED+n)\n    elapsed=time.perf_counter()-st\n    low,high=wilson_score(result[\'success_count\'],n)\n    mc_rows.append((n,result[\'success_count\'],result[\'availability\'],low,high,exact,\n                    result[\'mean_received_mw\'],elapsed))\ncsvout(\'05_monte_carlo_verification.csv\',[\'samples\',\'successes\',\'sample_availability\',\'Wilson95_lower\',\'Wilson95_upper\',\'exact_availability\',\'sample_mean_power_mw\',\'elapsed_seconds\'],mc_rows)\nplt.figure(figsize=(7.8,4.4))\nfor n,k,hat,lo,hi,an,mw,dt in mc_rows:\n    plt.errorbar([n],[hat],yerr=[[hat-lo],[hi-hat]],fmt=\'o\',capsize=5)\nplt.axhline(exact,color=\'black\',ls=\'--\',label=f\'Exact: {exact:.4f}\')\nplt.xscale(\'log\');plt.xlabel(\'Monte Carlo samples (log scale)\')\nplt.ylabel(\'Optical-power availability\');plt.title(\'08 | Independent Monte Carlo versus exact probability\')\nplt.legend();savefig(\'08_monte_carlo_validation\')\n\n# 06 extra: illustrative alternative priors (total sum=1 each, not measured)\nalt_priors={\'baseline_illustrative\':DEFAULT_PRIOR,\n            \'more_fog\':np.array([.68,.10,.15,.04,.03]),\n            \'more_rain\':np.array([.72,.10,.07,.08,.03]),\n            \'favorable_clear\':np.array([.84,.06,.04,.04,.02])}\nprior_rows=[]\nfor label, prior in alt_priors.items():\n    for rho in [0,.04]:\n        prob=availability(.14,15,link,prior,rho)\n        needed=min_aperture_for_target(TARGET,15,link,prior,rho)\n        prior_rows.append((label,*[round(float(v),5) for v in prior],rho,prob,float(needed) if np.isfinite(needed) else None))\ncsvout(\'06_alternate_illustrative_weather_priors.csv\',\n       [\'prior_label\',*[f\'p_{x}\' for x in SCENARIO_NAMES],\'TV_radius\',\'availability_at_radius_14cm\',\'minimum_radius_m_target_92pct\'],prior_rows)\n\n# 09: legacy atmospheric ceiling / global feasibility certificate\n# Retain this plot for reproducibility of the earlier exponential case only.\nlegacy=replace(link,convention=PAPER_EXPONENTIAL)\nrho_l=np.linspace(0,.11,90)\ncaps=np.array([weather_limited_ceiling(legacy,rho=float(z))[0] for z in rho_l])\ncritical=critical_tv_radius(TARGET,legacy)\nrows=[(float(r),float(v),int(v>=TARGET)) for r,v in zip(rho_l,caps)]\ncsvout(\'07_atmospheric_feasibility_ceiling.csv\',\n       [\'TV_radius\',\'global_atmosphere_limited_availability_ceiling\',\'target_92pct_not_excluded\'],rows)\nplt.figure(figsize=(8,4.7))\nplt.plot(rho_l,caps,label=\'Maximum possible modeled availability (infinite aperture)\')\nplt.axhline(TARGET,color=\'black\',ls=\':\',label=\'Target 92%\')\nplt.axvline(critical,color=\'darkred\',ls=\'--\',label=f\'Break-even TV radius {critical:.4f}\')\nplt.fill_between(rho_l,caps,TARGET,where=(caps<TARGET),color=\'red\',alpha=.12,label=\'Globally infeasible region\')\nplt.xlabel(\'Weather-probability total-variation radius\')\nplt.ylabel(\'Maximum availability permitted by atmospheric transmission\')\nplt.ylim(.82,1.0)\nplt.title(\'09 | Legacy exponential convention: atmospheric ceiling\')\nplt.legend(loc=\'lower left\',fontsize=8)\nsavefig(\'09_atmospheric_feasibility_ceiling\')\n\n\n# 10: Mandatory end-to-end convention sensitivity of the optimization itself.\n# The published coefficient table says dB/km while the equation uses exp(-Q).\n# Main physics/design conclusions must be conditional on this unresolved choice.\nconvention_rows=[]\nfor conv in (PAPER_EXPONENTIAL,DB_LABEL):\n    c_link=replace(link,convention=conv)\n    for rho in (0., .04, .08):\n        radius=min_aperture_for_target(TARGET,15,c_link,rho=rho)\n        ceiling,_=weather_limited_ceiling(c_link,rho=rho)\n        convention_rows.append((conv,rho,200*radius if np.isfinite(radius) else None,\n            float(ceiling),bool(ceiling + 1e-10 < TARGET)))\ncsvout(\'08_optimization_convention_sensitivity.csv\',\n       [\'coefficient_convention\',\'TV_radius\',\'minimum_aperture_diameter_cm_at_15urad\',\n        \'atmosphere_only_availability_ceiling\',\'globally_infeasible\'],convention_rows)\nfig,ax=plt.subplots(1,2,figsize=(11.0,4.5))\nfor c,marker in ((PAPER_EXPONENTIAL,\'o\'),(DB_LABEL,\'s\')):\n    ss=[x for x in convention_rows if x[0]==c]\n    radii=[float(\'nan\') if x[2] is None else x[2] for x in ss]\n    ax[0].plot([x[1]*100 for x in ss],radii,marker=marker,label=(\'Equation as printed: exp(-Q)\' if c==PAPER_EXPONENTIAL else \'Literal label: 10^(-Q/10)\'))\n    ax[1].plot([x[1]*100 for x in ss],[x[3] for x in ss],marker=marker,label=(\'Equation as printed\' if c==PAPER_EXPONENTIAL else \'Literal dB/km\'))\nax[0].set(xlabel=\'TV probability shift (%)\',ylabel=\'Minimum aperture diameter (cm)\',title=\'Required aperture for a 92% target\')\nax[1].set(xlabel=\'TV probability shift (%)\',ylabel=\'Atmosphere-only maximum availability\',title=\'Model feasibility bound\')\nax[1].axhline(TARGET,color=\'black\',ls=\':\',label=\'92% target\')\nfor a in ax: a.grid(alpha=.2);a.legend(fontsize=7)\nfig.suptitle(\'10 | Main engineering conclusions depend on the coefficient convention\',fontsize=11)\nsavefig(\'10_optimization_convention_sensitivity\')\n\n# 11: independent model stress cases and several availability targets.\nprofiles=[(\'table_ii_db\',link),(\'table_ii_legacy\',legacy),\n          (\'clear_prose_db\',replace(link,profile=\'clear_prose\')),\n          (\'fog_visibility_db\',replace(link,profile=\'fog_visibility_stress\'))]\nmodel_rows=[]\nfor name,cl in profiles:\n    for target in [.90,.92,.95,.99,.999]:\n        for rho in [0.,.04,.08]:\n            ceiling,_=weather_limited_ceiling(cl,rho=rho)\n            a=min_aperture_for_target(target,15,cl,rho=rho)\n            status=(\'feasible_in_interval\' if np.isfinite(a) else\n                    \'ceiling_excludes_target\' if ceiling<target else \'not_feasible_in_interval\')\n            achieved=availability(a,15,cl,rho=rho) if np.isfinite(a) else None\n            model_rows.append((name,target,rho,200*a if np.isfinite(a) else None,ceiling,achieved,status))\n            if np.isfinite(a): assert achieved>=target\ncsvout(\'09_model_and_target_sensitivity.csv\',[\'profile\',\'target\',\'tv_radius\',\'diameter_cm\',\'finite_aperture_ceiling\',\'achieved_availability\',\'status\'],model_rows)\nfig,ax=plt.subplots(1,2,figsize=(10,4.5))\nfor name,cl in profiles:\n    rs=[r for r in model_rows if r[0]==name and r[1]==.92]\n    ax[0].plot([r[2]*100 for r in rs],[np.nan if r[3] is None else r[3] for r in rs],marker=\'o\',label=name)\n    ax[1].plot([r[2]*100 for r in rs],[r[4] for r in rs],marker=\'o\',label=name)\nax[0].set(xlabel=\'TV radius (%)\',ylabel=\'Minimum diameter (cm)\',title=\'92% target: model sensitivity\')\nax[1].set(xlabel=\'TV radius (%)\',ylabel=\'Atmospheric upper bound\',title=\'Feasibility bound\')\nax[1].axhline(.92,color=\'black\',ls=\':\')\nfor a in ax:a.legend(fontsize=7);a.grid(alpha=.2)\nsavefig(\'11_model_and_target_sensitivity\')\n\n# 12: comparison against a separate published radiation-fog model, not measurement.\nvis=np.array([.05,.20,.50,.77])\npublished=np.array([7.0721,1.7680,.7072,.4592])\nexternal=np.array([radiation_fog_db_per_km(v) for v in vis])\ncsvout(\'10_fog_model_benchmark.csv\',[\'visibility_km\',\'table_I_label_db_per_km\',\'legacy_effective_db_per_km\',\'radiation_fog_native_extinction_per_km\',\'radiation_fog_converted_db_per_km\'],zip(vis,published,published*10/np.log(10),external*np.log(10)/10,external))\nplt.figure(figsize=(7,4.5));plt.loglog(vis,published,\'o-\',label=\'Table I literal dB/km\');plt.loglog(vis,published*10/np.log(10),\'s--\',label=\'Legacy exponential equivalent dB/km\');plt.loglog(vis,external,\'^-\',label=\'Radiation fog: converted dB/km\');plt.loglog(vis,external*np.log(10)/10,\'x:\',label=\'Radiation polynomial as dB (diagnostic)\')\nplt.xlabel(\'Visibility (km)\');plt.ylabel(\'Specific attenuation (dB/km)\');plt.title(\'12 | Fog model discrepancy (not measurement validation)\');plt.legend(fontsize=8);plt.grid(alpha=.2);savefig(\'12_fog_model_benchmark\')\n\n# 13: multi-geometry, multi-seed numerical verification, with analytic mean-power check.\nmc_multi=[]\nfor name,cl in [(\'db_vertical\',link),(\'legacy_vertical\',legacy),(\'db_slant\',replace(link,altitude_km=2.,zenith_deg=30.)),(\'fog_stress\',replace(link,profile=\'fog_visibility_stress\'))]:\n    a=.08; j=15.; ex=availability(a,j,cl)\n    tmean=sum(p*float(np.dot(pr,transmittance_from_q(q,cl.convention))) for p,scenario in zip(DEFAULT_PRIOR,SCENARIO_NAMES) for q,pr in [exact_distribution(scenarios_1550nm(cl.profile)[scenario],cl.altitude_km,cl.zenith_deg)])\n    mean=cl.tx_mw*cl.optical_efficiency*tmean*mean_capture_analytic(a,cl.beam_radius_m,j*1e-6*cl.length_m)\n    for seed in [SEED+11,SEED+22,SEED+33]:\n        n=20000 if FAST else 100000\n        out=sample_link(n,a,j,cl,seed=seed); lo,hi=wilson_score(out[\'success_count\'],n)\n        err=abs(out[\'availability\']-ex); se=math.sqrt(ex*(1-ex)/n)\n        power_se=out[\'std_received_mw\']/math.sqrt(n)\n        # Conservative simultaneous diagnostic tolerance; not a field-validation claim.\n        assert err<=5*se+5/n\n        assert abs(out[\'mean_received_mw\']-mean)<=5*power_se+1e-9\n        mc_multi.append((name,seed,n,ex,out[\'availability\'],lo,hi,mean,out[\'mean_received_mw\'],power_se,err))\ncsvout(\'11_multi_case_mc.csv\',[\'case\',\'seed\',\'n\',\'analytic_availability\',\'mc_availability\',\'wilson95_lower\',\'wilson95_upper\',\'analytic_mean_mw\',\'mc_mean_mw\',\'mc_mean_standard_error_mw\',\'absolute_availability_error\'],mc_multi)\n\n# Runtime and reproducibility information\nconfig={\n    \'paper_source\':\'https://doi.org/10.1109/MVT.2026.3687349\',\n    \'arxiv_manuscript\':\'https://arxiv.org/abs/2604.18207\',\n    \'paper_table_II_finite_states\':True,\n    \'weather_scenario_priors_empirically_estimated\':False,\n    \'weather_prior_order\':list(SCENARIO_NAMES),\n    \'weather_prior_values\':DEFAULT_PRIOR.tolist(),\n    \'reference_link\':asdict(link),\n    \'target_availability\':TARGET,\n    \'tv_radius\':RHO,\n    \'monte_carlo_seed_base\':SEED,\n    \'fast_mode\':FAST,\n    \'monte_carlo_samples_max\':N_MC,\n    \'method\':\'exact conditional mixture + Rayleigh jitter CDF + TV mass transfer; independent scipy linprog and random draws\',\n    \'receiver_power_threshold_is_illustrative_not_device_data\':True,\n    \'propagation_convention_primary\':DB_LABEL,\n    \'legacy_convention_is_source_comparison_only\':True,\n    \'implied_gaussian_equivalent_m_squared\':link.implied_m_squared,\n    \'minimum_aperture_search_radius_m\':[.025,.50],\n    \'source_profiles\':[name for name,_ in profiles],\n    \'end_to_end_sensitivity_in_both_conventions\':True,\n    \'note\':\'Only single-link optical-power availability; not BER or field reliability.\'\n}\n(META/\'run_config.json\').write_text(json.dumps(config,indent=2),encoding=\'utf8\')\n(META/\'environment.json\').write_text(json.dumps({\'python\':sys.version.split()[0],\n    \'numpy\':np.__version__,\'scipy\':scipy.__version__,\'matplotlib\':matplotlib.__version__,\n    \'platform\':platform.platform(),\'os_cpu_count\':os.cpu_count()},indent=2),encoding=\'utf8\')\n# Explicit timing measures end-to-end including plotting and storage, not estimated Colab\nrun_secs=time.perf_counter()-start\n(META/\'runtime.json\').write_text(json.dumps({\'wall_seconds_end_to_end_approx\':run_secs,\n\'host\':\'machine on which this script was executed (see environment.json)\',\'fast_mode\':FAST,\'mc_max_samples\':N_MC,\n\'peak_process_rss_mib_linux\':(__import__(\'resource\').getrusage(__import__(\'resource\').RUSAGE_SELF).ru_maxrss/1024 if sys.platform.startswith(\'linux\') else None)},indent=2),encoding=\'utf8\')\npaths=sorted(p for p in OUT.rglob(\'*\') if p.is_file() and p.name!=\'manifest.json\' and not p.name.startswith(\'.\'))\nmanifest={\'source_sha256\':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in [ROOT/\'src/atmosphere.py\',ROOT/\'src/fso_link.py\',ROOT/\'run_study.py\',ROOT/\'tests/test_link.py\',ROOT/\'tests/test_audit.py\'] if p.exists()},\'files\':[{\'path\':str(p.relative_to(OUT)), \'bytes\':p.stat().st_size,\n    \'sha256\':hashlib.sha256(p.read_bytes()).hexdigest()} for p in paths]}\n(META/\'manifest.json\').write_text(json.dumps(manifest,indent=2),encoding=\'utf8\')\nprint(\'RESULTS_READY\',len(list(FIG.glob(\'*.png\'))),\'plots /\',len(list(TBL.glob(\'*.csv\'))),\'tables\',\n      \'runtime\',round(run_secs,2),\'s\', \'exact=\',round(exact,5))\nprint(\'Pointing jitter 15 µrad: min diameter nom / robust (cm):\',\n      round(200*min_aperture_for_target(TARGET,15,link,rho=0),2),\n      round(200*min_aperture_for_target(TARGET,15,link,rho=RHO),2))\nprint(\'Legacy exponential model break-even TV radius:\',round(critical,6))\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `compare_results.py`

In [ ]:
p = PROJECT_ROOT / 'compare_results.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Compare regenerated FSO numerical results against reference CSV tables."""\n\nimport argparse\nimport csv\nimport math\nfrom pathlib import Path\n\nRTOL = 1e-8\nATOL = 1e-9\n\nEXACT_FIELDS = {\n    "seed", "n", "samples", "successes", "finite_states"\n}\n\ndef equal_value(expected, actual, field):\n\n    if expected == actual:\n        return True\n\n    try:\n        x = float(expected)\n        y = float(actual)\n    except ValueError:\n        return False\n\n    if math.isnan(x) or math.isnan(y):\n        return math.isnan(x) and math.isnan(y)\n\n    if field.lower() in EXACT_FIELDS:\n        return x == y\n\n    return math.isclose(\n        x, y, rel_tol=RTOL, abs_tol=ATOL\n    )\n\n\ndef read_csv(path):\n\n    with path.open(newline="", encoding="utf-8") as f:\n        rows = list(csv.reader(f))\n\n    if not rows:\n        raise ValueError(f"Empty CSV: {path}")\n\n    header, data = rows[0], rows[1:]\n\n    if len(header) != len(set(header)):\n        raise ValueError(f"Duplicate columns: {path}")\n\n    if any(len(row) != len(header) for row in data):\n        raise ValueError(f"Malformed CSV rows: {path}")\n\n    return header, data\n\n\ndef compare(reference, generated):\n\n    expected_files = {\n        p.name for p in reference.glob("*.csv")\n    }\n\n    actual_files = {\n        p.name for p in generated.glob("*.csv")\n    }\n\n    if expected_files != actual_files:\n        raise AssertionError(\n            f"Missing: {expected_files - actual_files}; "\n            f"Unexpected: {actual_files - expected_files}"\n        )\n\n    if len(expected_files) != 11:\n        raise AssertionError(\n            f"Expected 11 tables, found {len(expected_files)}"\n        )\n\n    failures = []\n    total_rows = 0\n    total_cells = 0\n\n    for name in sorted(expected_files):\n\n        header1, rows1 = read_csv(reference / name)\n        header2, rows2 = read_csv(generated / name)\n\n        if header1 != header2:\n            failures.append(f"{name}: column mismatch")\n            continue\n\n        if len(rows1) != len(rows2):\n            failures.append(f"{name}: row count mismatch")\n            continue\n\n        file_failures = 0\n\n        for index, (row1, row2) in enumerate(\n            zip(rows1, rows2), start=2\n        ):\n            total_rows += 1\n\n            for field, expected, actual in zip(\n                header1, row1, row2\n            ):\n\n                if field == "elapsed_seconds":\n                    continue\n\n                total_cells += 1\n\n                if not equal_value(expected, actual, field):\n                    failures.append(\n                        f"{name}, row {index}, {field}: "\n                        f"expected {expected}, obtained {actual}"\n                    )\n                    file_failures += 1\n\n        status = "PASS" if file_failures == 0 else "FAIL"\n\n        print(\n            f"{status}: {name} ({len(rows1)} rows)"\n        )\n\n    print("\\nVerification summary")\n    print("Tables:", len(expected_files))\n    print("Rows compared:", total_rows)\n    print("Cells compared:", total_cells)\n    print("Discrepancies:", len(failures))\n\n    if failures:\n        for failure in failures[:20]:\n            print(failure)\n\n        raise SystemExit("NUMERICAL REGRESSION FAILED")\n\n    print("ALL NUMERICAL REGRESSION CHECKS PASSED")\n\n\nif __name__ == "__main__":\n\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument(\n        "reference", type=Path\n    )\n\n    parser.add_argument(\n        "generated", type=Path\n    )\n\n    args = parser.parse_args()\n\n    compare(args.reference, args.generated)\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `tests/test_link.py`

In [ ]:
p = PROJECT_ROOT / 'tests/test_link.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("import math,sys,unittest\nfrom pathlib import Path\nimport numpy as np\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/'src'))\nfrom atmosphere import (PAPER_EXPONENTIAL,DB_LABEL,scenarios_1550nm,\n                        exact_distribution,transmittance_from_q)\nfrom fso_link import (Link,DEFAULT_PRIOR,SCENARIO_NAMES,gaussian_aperture_capture,\n    capture_via_independent_quadrature,mean_capture_analytic,critical_offset,\n    success_probability_for_transmittance,conditional_availability,availability,\n    worst_case_tv_greedy,worst_case_tv_lp,min_aperture_for_target,sample_link,\n    wilson_score,weather_limited_ceiling,critical_tv_radius)\n\nclass PhysicalAndNumericalChecks(unittest.TestCase):\n    def setUp(self): self.link=Link(convention=PAPER_EXPONENTIAL)\n    def test_01_source_state_probabilities(self):\n        for name,s in scenarios_1550nm().items():\n            self.assertEqual(s.name,scenarios_1550nm()[name].name)\n            for l in s.layers:self.assertAlmostEqual(sum(st.probability for st in l.states),1)\n    def test_02_zero_offset_closed_form(self):\n        for a,w in [(.1,.2),(.2,.18),(.03,.5)]:\n            self.assertAlmostEqual(gaussian_aperture_capture(a,w,0),1-math.exp(-2*a*a/(w*w)),places=11)\n    def test_03_independent_bessel_quadrature(self):\n        for a,w,r in [(.1,.17,0),(.1,.17,.02),(.1,.17,.13),(.25,.18,.3)]:\n            self.assertAlmostEqual(gaussian_aperture_capture(a,w,r),capture_via_independent_quadrature(a,w,r),places=9)\n    def test_04_capture_decreases_with_offset(self):\n        v=gaussian_aperture_capture(.15,.18,np.linspace(0,.6,100))\n        self.assertTrue(np.all(np.diff(v)<=1e-11))\n    def test_05_capture_increases_with_aperture(self):\n        self.assertLess(gaussian_aperture_capture(.1,.2,.1),gaussian_aperture_capture(.2,.2,.1))\n    def test_06_critical_displacement_inversion(self):\n        target=.25; r=critical_offset(.12,.18,target)\n        self.assertGreater(r,0)\n        self.assertAlmostEqual(gaussian_aperture_capture(.12,.18,r),target,places=9)\n    def test_07_analytic_mean_capture_convolution(self):\n        a,w,s=.14,.2,.065\n        mean=mean_capture_analytic(a,w,s)\n        rng=np.random.default_rng(2026)\n        r=np.hypot(rng.normal(0,s,80000),rng.normal(0,s,80000))\n        simulated=np.mean(gaussian_aperture_capture(a,w,r))\n        self.assertLess(abs(mean-simulated),.003)\n    def test_08_zero_jitter_conditional(self):\n        for t in (.001,.5,1):\n            ans=success_probability_for_transmittance(t,.15,0,self.link)\n            beam=gaussian_aperture_capture(.15,self.link.beam_radius_m,0)\n            expected=float(self.link.tx_mw*self.link.optical_efficiency*t*beam>=self.link.threshold_mw)\n            self.assertEqual(ans,expected)\n    def test_09_independent_rayleigh_pointing_verification(self):\n        t=.35;a=.12;j=22\n        exact=success_probability_for_transmittance(t,a,j,self.link)\n        rng=np.random.default_rng(31);n=100000\n        sd=j*1e-6*self.link.length_m\n        radial=np.hypot(rng.normal(0,sd,n),rng.normal(0,sd,n))\n        mc=(self.link.tx_mw*self.link.optical_efficiency*t*gaussian_aperture_capture(a,self.link.beam_radius_m,radial)>=self.link.threshold_mw).mean()\n        self.assertLess(abs(exact-mc),.008)\n    def test_10_conditional_probabilities_valid(self):\n        v=conditional_availability(.14,15,self.link)\n        self.assertEqual(len(v),5)\n        self.assertTrue(np.all((v>=0)&(v<=1)))\n    def test_11_prior_and_nominal(self):\n        v=conditional_availability(.14,15,self.link)\n        self.assertAlmostEqual(availability(.14,15,self.link),np.dot(DEFAULT_PRIOR,v),places=10)\n    def test_12_robust_exact_lp(self):\n        rng=np.random.default_rng(6)\n        for _ in range(65):\n            p=rng.dirichlet(np.ones(5));v=rng.random(5);rho=float(rng.random())\n            self.assertAlmostEqual(worst_case_tv_greedy(v,p,rho),worst_case_tv_lp(v,p,rho),places=8)\n    def test_13_robust_is_not_optimistic(self):\n        nom=availability(.14,15,self.link)\n        self.assertLessEqual(availability(.14,15,self.link,rho=.04),nom)\n        self.assertLessEqual(availability(.14,15,self.link,rho=.08),availability(.14,15,self.link,rho=.04))\n    def test_14_minimum_aperture_constraint(self):\n        r=min_aperture_for_target(.92,15,self.link,rho=.04)\n        self.assertTrue(.10<r<.20)\n        self.assertAlmostEqual(availability(r,15,self.link,rho=.04),.92,delta=.0003)\n        self.assertLess(availability(r-.002,15,self.link,rho=.04),.92)\n    def test_15_unachievable_target_flag(self):\n        self.assertTrue(math.isnan(min_aperture_for_target(.92,15,self.link,rho=.08)))\n    def test_16_stochastic_cross_check(self):\n        ex=availability(.14,15,self.link)\n        run=sample_link(25000,.14,15,self.link,seed=987)\n        self.assertLess(abs(ex-run['availability']),.014)\n    def test_17_receiver_power_threshold_monotone(self):\n        from dataclasses import replace\n        l2=replace(self.link,threshold_mw=self.link.threshold_mw*2)\n        self.assertGreaterEqual(availability(.15,15,self.link),availability(.15,15,l2))\n    def test_18_different_conventions(self):\n        from dataclasses import replace\n        l2=replace(self.link,convention=DB_LABEL)\n        v1=availability(.09,15,self.link);v2=availability(.09,15,l2)\n        self.assertNotAlmostEqual(v1,v2,places=3)\n    def test_19_wilson(self):\n        lo,hi=wilson_score(5,10)\n        self.assertTrue(0<lo<.5<hi<1)\n    def test_21_global_availability_ceiling(self):\n        c0,s=weather_limited_ceiling(self.link,rho=0)\n        self.assertAlmostEqual(c0,.964,places=12)\n        self.assertTrue(np.all((s>=0)&(s<=1)))\n        self.assertGreaterEqual(c0,availability(.30,15,self.link))\n    def test_22_tv_feasibility_threshold(self):\n        r=critical_tv_radius(.92,self.link)\n        self.assertAlmostEqual(r,(.964-.92)/.9,places=10)\n        self.assertLess(weather_limited_ceiling(self.link,rho=.08)[0],.92)\n        self.assertGreater(weather_limited_ceiling(self.link,rho=.04)[0],.92)\n    def test_23_coefficient_convention_end_to_end_conclusion_changes(self):\n        from dataclasses import replace\n        a=replace(self.link,convention=DB_LABEL)\n        self.assertTrue(math.isinf(critical_tv_radius(.92,a)))\n        self.assertAlmostEqual(weather_limited_ceiling(a,rho=.08)[0],1.0)\n        self.assertTrue(math.isfinite(min_aperture_for_target(.92,15,a,rho=.08)))\n        self.assertTrue(math.isnan(min_aperture_for_target(.92,15,self.link,rho=.08)))\n    def test_24_literal_db_implies_less_attenuation_for_positive_coefficients(self):\n        import numpy as np\n        from atmosphere import transmittance_from_q, PAPER_EXPONENTIAL, DB_LABEL\n        q=np.array([0.03,.1,1.,5.,10.])\n        self.assertTrue(np.all(transmittance_from_q(q,DB_LABEL)>transmittance_from_q(q,PAPER_EXPONENTIAL)))\n    def test_20_weather_probabilities_across_scenarios_not_sourced(self):\n        self.assertAlmostEqual(DEFAULT_PRIOR.sum(),1.)\n        self.assertEqual(len(SCENARIO_NAMES),len(DEFAULT_PRIOR))\n\nif __name__=='__main__':unittest.main(verbosity=2)\n", encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `tests/test_audit.py`

In [ ]:
p = PROJECT_ROOT / 'tests/test_audit.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Regression tests independent of the original portfolio\'s test suite."""\nimport sys, math, unittest\nfrom pathlib import Path\nfrom dataclasses import replace\nimport numpy as np\nfrom scipy.integrate import quad\nfrom scipy.optimize import brentq\nfrom scipy.special import erf\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'src\'))\nfrom atmosphere import (DB_LABEL,PAPER_EXPONENTIAL,scenarios_1550nm,exact_distribution,\n    path_lengths_km,transmittance_from_q,loss_db_from_q,effective_transmittance,\n    exact_statistics,mean_transmittance_factorized,radiation_fog_db_per_km)\nfrom fso_link import (Link, DEFAULT_PRIOR, gaussian_aperture_capture, availability,\n    conditional_availability, min_aperture_for_target, worst_case_tv_greedy,\n    worst_case_tv_lp, weather_limited_ceiling, sample_link, success_probability_for_transmittance)\n\n# Independent Table II fixtures, checked against arXiv:2604.18207v1, Table II\n# and Section IV-A (cloud placement). Units reproduce printed values.\nFIXTURES={\n\'clear\':[(0,3,[(1,.034)]),(3,15,[(1,.0025)])],\n\'pollution\':[(0,3,[(.7,.3536),(.3,.034)]),(3,15,[(1,.0025)])],\n\'fog\':[(0,1,[(1,1.768)]),(1,2,[(1,.4592)]),(2,15,[(1,.0025)])],\n\'rain\':[(0,.6,[(1,.0025)]),(.6,1.4,[(.9,8.2425),(.1,.034)]),(1.4,15,[(1,.0025)])],\n\'snow\':[(0,2,[(1,.2)]),(2,15,[(1,.0025)])]}\nVOLCANIC=(15,30,[(.5,.0104),(.5,.0002036)])\n# Hand-computed 3km integrated printed attenuation and probability pairs.\nREF=[[(.102,1)],[(1.0608,.7),(.102,.3)],[(2.2297,1)],[(6.5995,.9),(.0327,.1)],[(.4025,1)]]\n\ndef independent_capture(a,w,r):\n    # Cartesian slicing, not Bessel integration or noncentral chi-squared CDF.\n    scale=math.sqrt(2)/w\n    def f(x):\n        y=math.sqrt(max(0.,a*a-x*x))\n        return math.sqrt(2/math.pi)/w*math.exp(-2*((x-r)/w)**2)*erf(scale*y)\n    return quad(f,-a,a,epsabs=2e-12,epsrel=2e-11)[0]\n\ndef independent_probability(a,j,link,rho):\n    # Independent state fixture, explicit power convention, Cartesian integral.\n    w=math.hypot(.025,55e-6*3000);sd=j*1e-6*3000\n    vec=[]\n    for states in REF:\n        p=0.\n        for q,prob in states:\n            t=10**(-q/10) if link.convention==DB_LABEL else math.exp(-q)\n            needed=2/(35*t)\n            if needed>=independent_capture(a,w,0): continue\n            rc=brentq(lambda r: independent_capture(a,w,r)-needed,0,a+10*w,xtol=1e-12)\n            p+=prob*(-math.expm1(-rc**2/(2*sd**2)))\n        vec.append(p)\n    return worst_case_tv_lp(vec,np.array([.76,.10,.07,.04,.03]),rho)\n\nclass AuditChecks(unittest.TestCase):\n    def test_source_fixture_all_layers(self):\n        for name,expected in FIXTURES.items():\n            actual=[(l.bottom_km,l.top_km,[(s.probability,s.printed_coefficient_per_km) for s in l.states]) for l in scenarios_1550nm()[name].layers]\n            self.assertEqual(actual,expected+[VOLCANIC])\n    def test_reference_state_values(self):\n        for name,expected in zip((\'clear\',\'pollution\',\'fog\',\'rain\',\'snow\'),REF):\n            q,p=exact_distribution(scenarios_1550nm()[name],3)\n            np.testing.assert_allclose(q,[x[0] for x in expected],atol=1e-12)\n            np.testing.assert_allclose(p,[x[1] for x in expected],atol=1e-12)\n    def test_db_unit_conversion(self):\n        self.assertEqual(Link().convention,DB_LABEL)\n        self.assertAlmostEqual(transmittance_from_q(10),.1)\n        self.assertAlmostEqual(transmittance_from_q(3),10**(-.3))\n        self.assertAlmostEqual(float(loss_db_from_q(2,PAPER_EXPONENTIAL)),20/math.log(10))\n    def test_layer_geometry_boundaries(self):\n        for name,s in scenarios_1550nm().items():\n            for h in [0,.3,.6,1,1.4,2,3,15,20,30]:\n                for z in [0,30,70]:\n                    self.assertAlmostEqual(sum(path_lengths_km(s,h,z)),h/math.cos(math.radians(z)),places=10)\n    def test_jensen_and_factorization(self):\n        for name,s in scenarios_1550nm().items():\n            for conv in [DB_LABEL,PAPER_EXPONENTIAL]:\n                result=exact_statistics(s,25,45,convention=conv)\n                self.assertGreaterEqual(result[\'mean_transmittance\']+1e-14,result[\'effective_transmittance\'])\n                self.assertAlmostEqual(result[\'mean_transmittance\'],mean_transmittance_factorized(s,25,45,conv),places=12)\n    def test_prose_and_fog_stress(self):\n        q,p=exact_distribution(scenarios_1550nm(\'clear_prose\')[\'clear\'],3)\n        self.assertAlmostEqual(q[0],.0075)\n        self.assertAlmostEqual(radiation_fog_db_per_km(.2),(10/math.log(10))*(.18126*1.55**2+.13709*1.55+3.7502)/.2)\n        q,p=exact_distribution(scenarios_1550nm(\'fog_visibility_stress\')[\'fog\'],3)\n        self.assertGreater(q[0],20)\n        with self.assertRaises(ValueError): Link(profile=\'fog_visibility_stress\',convention=PAPER_EXPONENTIAL)\n    def test_cartesian_aperture(self):\n        for a,w,r in [(.025,.16688,0),(.1,.2,.07),(.14,.17,.25),(.25,.18,.3)]:\n            self.assertAlmostEqual(float(gaussian_aperture_capture(a,w,r)),independent_capture(a,w,r),places=10)\n    def test_independent_reference_design(self):\n        for conv in [DB_LABEL,PAPER_EXPONENTIAL]:\n            l=Link(convention=conv)\n            for rho in [0,.04,.08]:\n                a=min_aperture_for_target(.92,15,l,rho=rho)\n                if math.isfinite(a):\n                    self.assertGreaterEqual(availability(a,15,l,rho=rho),.92)\n                    self.assertAlmostEqual(independent_probability(a,15,l,rho),.92,delta=4e-6)\n                    self.assertLess(independent_probability(a-2e-7,15,l,rho),.92)\n                else: self.assertLess(weather_limited_ceiling(l,rho=rho)[0],.92)\n    def test_zero_jitter_step_boundary(self):\n        l=Link();a=min_aperture_for_target(.92,0,l)\n        self.assertGreaterEqual(availability(a,0,l),.92)\n        self.assertLess(availability(a-2e-7,0,l),.92)\n    def test_exact_threshold_equality(self):\n        # 35 mW equals the *unattenuated* limiting power. Finite Gaussian capture < 1.\n        l=replace(Link(),threshold_mw=35.)\n        self.assertEqual(success_probability_for_transmittance(1,.5,0,l),0.)\n        self.assertEqual(success_probability_for_transmittance(1,.5,15,l),0.)\n        self.assertEqual(weather_limited_ceiling(l)[0],0.)\n    def test_bracket_tiny_negative_upper_residual(self):\n        l=Link();hi=.08;t=availability(hi,15,l)+1e-10\n        self.assertTrue(math.isnan(min_aperture_for_target(t,15,l,a_hi=hi)))\n    def test_nonfinite_inputs(self):\n        for bad in [float(\'nan\'),float(\'inf\')]:\n            with self.assertRaises(ValueError): gaussian_aperture_capture(.1,.1,bad)\n            with self.assertRaises(ValueError): success_probability_for_transmittance(bad,.1,15,Link())\n            with self.assertRaises(ValueError): worst_case_tv_greedy([bad,0,0,0,0],DEFAULT_PRIOR,.04)\n            with self.assertRaises(ValueError): sample_link(100,.1,bad,Link())\n        with self.assertRaises(ValueError): sample_link(1,.1,15,Link())\n        with self.assertRaises(ValueError): Link(divergence_urad=1.)\n    def test_weather_ceiling_stress(self):\n        l=Link(profile=\'fog_visibility_stress\')\n        self.assertAlmostEqual(weather_limited_ceiling(l)[0],.93)\n        self.assertAlmostEqual(weather_limited_ceiling(l,rho=.04)[0],.89)\n        self.assertTrue(math.isnan(min_aperture_for_target(.92,15,l,rho=.04)))\n    def test_tv_extremes(self):\n        a=np.array([.2,.5,.8,.1,.9]);q=DEFAULT_PRIOR\n        self.assertAlmostEqual(worst_case_tv_greedy(a,q,0),np.dot(a,q))\n        self.assertAlmostEqual(worst_case_tv_greedy(a,q,1),a.min())\n\n\nclass InputValidationRegressionTests(unittest.TestCase):\n\n    def test_attenuation_invalid_inputs(self):\n        for convention in [DB_LABEL, PAPER_EXPONENTIAL]:\n            for bad in [-1, float("nan"), float("inf"),\n                        -float("inf"), [0, -1]]:\n                with self.subTest(convention=convention, value=bad):\n                    with self.assertRaises(ValueError):\n                        loss_db_from_q(bad, convention)\n\n    def test_attenuation_valid_inputs(self):\n        self.assertAlmostEqual(loss_db_from_q(0), 0)\n        self.assertAlmostEqual(loss_db_from_q(10), 10)\n        self.assertAlmostEqual(\n            loss_db_from_q(1, PAPER_EXPONENTIAL),\n            10 / math.log(10)\n        )\n\n    def test_wilson_invalid_multiplier(self):\n        from fso_link import wilson_score\n\n        for bad in [-1.96, 0, float("nan"), float("inf")]:\n            with self.subTest(z=bad):\n                with self.assertRaises(ValueError):\n                    wilson_score(5, 10, bad)\n\n    def test_wilson_invalid_counts(self):\n        from fso_link import wilson_score\n\n        for count, n in [(-1, 10), (11, 10), (0, 0),\n                         (2.5, 10), (2, 10.5), (True, 10)]:\n            with self.subTest(count=count, n=n):\n                with self.assertRaises(ValueError):\n                    wilson_score(count, n)\n\n    def test_wilson_valid_interval(self):\n        from fso_link import wilson_score\n\n        lower, upper = wilson_score(5, 10)\n\n        self.assertAlmostEqual(lower, 0.236593, places=5)\n        self.assertAlmostEqual(upper, 0.763407, places=5)\n\n\nclass AtmosphericWilsonRegressionTests(unittest.TestCase):\n\n    def test_atmospheric_wilson_validation(self):\n        from atmosphere import wilson_interval\n        from fso_link import wilson_score\n\n        for bad in [-1, 0, float("nan"), float("inf")]:\n            with self.assertRaises(ValueError):\n                wilson_interval(5, 10, bad)\n\n        for successes, n in [\n            (-1, 10), (11, 10), (0, 0),\n            (2.5, 10), (2, 10.5), (True, 10)\n        ]:\n            with self.assertRaises(ValueError):\n                wilson_interval(successes, n)\n\n        for successes, n in [(0, 10), (5, 10), (10, 10)]:\n            np.testing.assert_allclose(\n                wilson_interval(successes, n),\n                wilson_score(successes, n),\n                rtol=1e-12,\n                atol=1e-12\n            )\n\nif __name__==\'__main__\': unittest.main()\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `README.md`

In [ ]:
p = PROJECT_ROOT / 'README.md'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('<h1 align="center">Atmospheric-Model Sensitivity in Ground-to-Air FSO Receiver Design</h1>\n\n<p align="center">\n  <strong>Nishat Yasmin Anika</strong><br>\n  Department of Electronics and Communication Engineering<br>\n  Khulna University of Engineering &amp; Technology (KUET), Bangladesh\n</p>\n\n<p align="center">\n  <a href="https://github.com/Nishat-Yasmin-Anika/ground-to-air-fso-link-design/actions/workflows/python-tests.yml"><img src="https://github.com/Nishat-Yasmin-Anika/ground-to-air-fso-link-design/actions/workflows/python-tests.yml/badge.svg?branch=main" alt="Numerical verification status"></a>\n  <a href="https://colab.research.google.com/github/Nishat-Yasmin-Anika/ground-to-air-fso-link-design/blob/main/notebooks/FSO_Link_Design_Colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>\n  <a href="LICENSE"><img src="https://img.shields.io/badge/License-MIT-blue.svg" alt="MIT license"></a>\n</p>\n\n<p align="center">\n  <a href="#abstract">Abstract</a> ·\n  <a href="#research-context-and-motivation">Motivation</a> ·\n  <a href="#model-and-method">Method</a> ·\n  <a href="#results-and-discussion">Results</a> ·\n  <a href="#numerical-verification">Verification</a> ·\n  <a href="#research-outlook">Outlook</a> ·\n  <a href="#reproduction">Reproduction</a> ·\n  <a href="#references">References</a>\n</p>\n\n<p align="center">\n  <img src="assets/cover.svg" width="100%" alt="Illustration of a ground-to-air optical link through atmospheric layers">\n</p>\n\n## Abstract\n\n<p align="justify">This numerical study explores how atmospheric attenuation assumptions influence receiver aperture sizing in ground-to-air free-space optical (FSO) links. It brings together altitude-dependent atmospheric layers, Gaussian beam collection through a circular aperture and random pointing errors. A total-variation (TV) ambiguity set is used to examine uncertainty in weather-scenario probabilities.</p>\n\n<p align="justify">For an illustrative 3 km vertical link with a 92% received-power availability target and 15 µrad per-axis angular jitter, the calculated nominal receiver diameter is 9.47 cm under the literal dB/km interpretation of the published attenuation coefficients and 21.95 cm under the exponential equation as printed in the author manuscript. At a TV radius of 0.08, the latter case gives an availability ceiling of 89.2%. Under those assumptions, no finite aperture can meet the target. Quadrature, linear programming and Monte Carlo simulation are used to check the numerical calculations.</p>\n\n<p align="justify">These results are conditional on the chosen model and parameters. The purpose is to understand the sensitivity of receiver design decisions, rather than to establish a measured link performance or a practical receiver specification.</p>\n\n<p align="justify"><strong>Keywords:</strong> Free-space optics, atmospheric attenuation, pointing errors, Gaussian beam collection, receiver aperture, distributionally robust optimization.</p>\n\n## Research context and motivation\n\n<p align="justify">I began this project to better understand atmospheric propagation and optical link design in airborne communication systems. The work of Erdogan et al. [1] on altitude-dependent atmospheric attenuation and the study by Elamassie and Uysal [4] on multi-layer airborne FSO systems provided the main motivation for this exploration. Reading these studies encouraged me to examine how atmospheric conditions and pointing uncertainty can influence the size of an optical receiver.</p>\n\n<p align="justify">The question I wanted to explore was whether a receiver sized for an assumed weather distribution would still meet its optical-power threshold if that distribution changed. I approached this question by implementing a simplified link model and examining the effects of its assumptions.</p>\n\n<p align="justify">While working through the atmospheric model in the author manuscript of [1], I was unsure how to interpret the coefficient convention. The tables label the coefficients in dB/km, while the transmittance equation uses a natural exponential. I considered both interpretations using the printed coefficient values to see how the receiver design would change. This comparison is intended as a sensitivity study, not a conclusion about the intended convention or the accuracy of the original atmospheric model. The underlying atmospheric calibration has not been reproduced.</p>\n\n<p align="justify">Two additional sensitivity cases consider the clear-weather layer described in the source text and a separate radiation-fog relation [2]. Established work on pointing errors [3] informs the optical model, while total variation [5] provides a way to represent uncertainty in the assumed weather probabilities.</p>\n\n<p align="justify">I see this as a learning-oriented computational study and a starting point for further research. Feedback on the modelling assumptions and their interpretation would be valuable.</p>\n\n## Model and method\n\n<p align="center">\n  <img src="assets/method.svg" width="100%" alt="Four stages of the model: atmospheric attenuation, Gaussian beam collection, optical-power availability and robust aperture sizing">\n</p>\n\n<p align="center"><em>Fig. 1. Calculation sequence for the receiver design problem.</em></p>\n\n### Atmospheric attenuation\n\n<p align="justify">For a receiver at altitude H and zenith angle θ, the slant path in the flat-layer approximation is H sec θ. Each intersected layer contributes its traversed distance to the integrated printed coefficient Q. The two power-transmittance interpretations are</p>\n\n```math\nQ=\\sum_j c_jd_j\n```\n\n```math\nT_{\\mathrm{dB}}=10^{-Q/10},\\qquad T_{\\mathrm{exp}}=e^{-Q}.\n```\n\n<p align="justify">Here, the layer distance d is measured in kilometres. The units of Q depend on the interpretation. The literal dB/km conversion is used as the primary case, while the exponential form is retained for comparison with the equation as printed in the author manuscript [1].</p>\n\n### Optical collection and pointing\n\n<p align="justify">Received optical power is modelled as the product of transmitted power, optical efficiency, atmospheric transmittance and the fraction collected by the circular aperture. Under the assumed Gaussian beam profile, the collection fraction can be expressed exactly using a noncentral chi-square cumulative distribution function with two degrees of freedom.</p>\n\n```math\nP_r=P_t\\eta T G(a,w,r)\n```\n\n```math\nG(a,w,r)=F_{\\chi\'^2_2}\\left(\\frac{4a^2}{w^2};\\frac{4r^2}{w^2}\\right).\n```\n\n<p align="justify">The variables a, w and r denote aperture radius, Gaussian beam radius and radial pointing displacement, respectively. Independent zero-mean Gaussian pointing components with equal variance give a Rayleigh distribution for r. For each atmospheric state, the maximum displacement satisfying the received-power threshold is found numerically.</p>\n\n### Aperture optimization\n\n<p align="justify">Let Aᵢ(a) be the availability conditional on weather scenario i. Let p be the nominal probability vector and q an alternative vector. The ambiguity set is</p>\n\n```math\n\\mathcal U_\\rho=\\left\\{q\\geq0:\\ \\sum_i q_i=1,\\quad \\frac12\\sum_i\\lvert q_i-p_i\\rvert\\leq\\rho\\right\\}.\n```\n\n<p align="justify">The minimum aperture is sought subject to the worst-case availability constraint:</p>\n\n```math\n\\begin{aligned}\n\\min_a\\quad &a \\\\\n\\text{subject to}\\quad &\\min_{q\\in\\mathcal U_\\rho}\\sum_i q_i A_i(a)\\geq A_{\\mathrm{target}},\\\\\n&0.025\\leq a\\leq0.50\\quad\\text{(metres)}.\n\\end{aligned}\n```\n\n<p align="justify">The inner problem is solved by transferring probability mass from scenarios with higher availability to those with lower availability. The outer search uses bisection because collection increases with aperture radius. An atmospheric availability ceiling helps distinguish targets that fail within the search interval from those that cannot be met at any finite aperture under the model.</p>\n\n<p align="center">\n  <a href="docs/MATHEMATICAL_DERIVATIONS.md">Mathematical derivations</a> ·\n  <a href="docs/TECHNICAL_NOTE.md">Technical note</a>\n</p>\n\n<details>\n<summary>Reference simulation parameters</summary>\n\n| Parameter | Assumed value |\n|---|---:|\n| Wavelength | 1550 nm |\n| Receiver altitude | 3 km |\n| Zenith angle | 0° |\n| Transmitted optical power | 50 mW |\n| Optical efficiency | 0.70 |\n| Beam waist radius | 25 mm |\n| Divergence half-angle | 55 µrad |\n| Per-axis angular jitter (standard deviation) | 15 µrad |\n| Received-power threshold | 2 mW |\n| Availability target | 0.92 |\n| Reference TV radius | 0.04 |\n\n<p align="justify">The assumed scenario probabilities for clear weather, pollution, fog, rain and snow are 0.76, 0.10, 0.07, 0.04 and 0.03, respectively. These are illustrative probabilities, not estimates from historical weather data. The received-power threshold is an assumed study parameter rather than a measured detector sensitivity.</p>\n\n</details>\n\n## Results and discussion\n\n<p align="justify">Table 1 reports the minimum receiver diameter for a 92% probability of meeting or exceeding the assumed 2 mW threshold in the reference link. All entries are model-based calculations.</p>\n\n<p align="center"><em>Table 1. Minimum receiver diameter (cm) under alternative atmospheric assumptions.</em></p>\n\n| Atmospheric case | Nominal | ρ = 0.04 | ρ = 0.08 |\n|---|---:|---:|---:|\n| Table II, literal dB/km | 9.47 | 11.82 | 13.62 |\n| Table II, exponential interpretation | 21.95 | 27.01 | Infeasible |\n| Clear-weather prose alternative | 9.42 | 11.75 | 13.62 |\n| Radiation-fog stress case | 15.65 | Infeasible | Infeasible |\n\n<p align="justify">Under literal dB/km conversion, increasing the TV radius from zero to 0.04 raises the calculated diameter from 9.47 to 11.82 cm. The corresponding change under the exponential interpretation is from 21.95 to 27.01 cm. At ρ = 0.08, the latter case has an availability ceiling of 89.2%, below the target. The radiation-fog stress case also falls below the target at ρ = 0.04, with a ceiling of 89%. Under the stated assumptions, these cases cannot meet 92% at any finite aperture.</p>\n\n<p align="center">\n  <a href="results/figures/11_model_and_target_sensitivity.svg"><img src="results/figures/11_model_and_target_sensitivity.png" width="100%" alt="Receiver diameter and atmospheric feasibility bounds for four atmospheric cases"></a>\n</p>\n\n<p align="center"><em>Fig. 2. Receiver aperture requirements and atmospheric feasibility bounds across four atmospheric models for a 92% availability target.</em> <a href="results/figures/11_model_and_target_sensitivity.svg">Vector figure</a></p>\n\n<p align="justify">The optical parameters remain fixed in Table 1. The differences therefore arise from the atmospheric assumptions and the uncertainty in weather probabilities rather than changes in transmitter power or beam geometry. The comparison helped me understand how strongly the receiver sizing calculation can depend on the atmospheric model. It does not establish which coefficient interpretation is physically correct.</p>\n\n<details>\n<summary>Additional results: pointing jitter and aperture size</summary>\n\n<p align="center">\n  <a href="results/figures/04_minimum_aperture_frontier.svg"><img src="results/figures/04_minimum_aperture_frontier.png" width="100%" alt="Minimum receiver diameter versus pointing angular jitter"></a>\n</p>\n\n<p align="center"><em>Fig. 3. Minimum receiver diameter versus per-axis angular jitter.</em> <a href="results/figures/04_minimum_aperture_frontier.svg">Vector figure</a></p>\n\n<p align="center">\n  <a href="results/figures/03_availability_vs_aperture.svg"><img src="results/figures/03_availability_vs_aperture.png" width="100%" alt="Nominal and robust received-power availability versus receiver aperture size"></a>\n</p>\n\n<p align="center"><em>Fig. 4. Received-power availability versus aperture size.</em> <a href="results/figures/03_availability_vs_aperture.svg">Vector figure</a></p>\n\n</details>\n\n<p align="center">\n  <a href="results/README.md">Complete results gallery (12 figures and 11 CSV tables)</a> ·\n  <a href="results/tables/09_model_and_target_sensitivity.csv">Source data for Table 1</a>\n</p>\n\n## Numerical verification\n\n<p align="justify">To check the implementation, the Gaussian aperture integral is evaluated using independent Bessel and Cartesian quadrature formulations. A separately formulated linear program checks the TV probability-transfer solution. Monte Carlo simulations draw weather states and Gaussian pointing displacements directly. Wilson intervals describe sampling uncertainty in the estimated success probabilities.</p>\n\n<p align="center">\n  <a href="results/figures/08_monte_carlo_validation.svg"><img src="results/figures/08_monte_carlo_validation.png" width="100%" alt="Monte Carlo estimates with Wilson intervals compared with analytic availability"></a>\n</p>\n\n<p align="center"><em>Fig. 5. Monte Carlo estimates and analytic availability.</em> <a href="results/figures/08_monte_carlo_validation.svg">Vector figure</a></p>\n\n<details>\n<summary>Independent aperture integral comparison</summary>\n\n<p align="center">\n  <a href="results/figures/01_aperture_integral_validation.svg"><img src="results/figures/01_aperture_integral_validation.png" width="100%" alt="Gaussian aperture collection calculated by independent numerical integration methods"></a>\n</p>\n\n<p align="center"><em>Fig. 6. Numerical agreement between aperture integration methods.</em> <a href="results/figures/01_aperture_integral_validation.svg">Vector figure</a></p>\n\n</details>\n\n<p align="justify">The version 4.1.0 test suite contains 44 tests. GitHub Actions regenerates the full experiments and compares the 11 numerical tables against the committed reference results, with execution time excluded. SHA-256 hashes check the integrity of the generated artifacts and generating source files. These checks support numerical consistency and reproducibility within the stated model. They do not constitute experimental validation of the optical link.</p>\n\n<p align="center">\n  <a href="docs/COMPLEXITY_AND_VERIFICATION.md">Verification methods and computational complexity</a> ·\n  <a href="https://github.com/Nishat-Yasmin-Anika/ground-to-air-fso-link-design/actions">Verification history</a>\n</p>\n\n## Limitations\n\n<p align="justify">The atmospheric coefficients have not been calibrated against local measurements. Weather probabilities, layer independence, optical parameters and the TV radius are assumptions. Turbulence, scintillation, receiver field-of-view effects, detector noise and temporal tracking dynamics are not included.</p>\n\n<p align="justify">Availability in this study refers to the probability of meeting a received-optical-power threshold. It is not measured uptime or bit-error-rate performance. The radiation-fog relation is used as a separate hypothetical sensitivity case and has not been validated for this airborne geometry. The results should therefore be interpreted within the stated assumptions rather than as general receiver design requirements.</p>\n\n## Research outlook\n\n<p align="justify">This work focuses on a single ground-to-air optical link. I would like to learn more about how altitude-dependent atmospheric conditions and platform motion influence pointing errors and the design of multi-layer airborne FSO networks. A possible next step would be to explore altitude selection and link optimization using more realistic atmospheric information.</p>\n\n<p align="justify">I am also interested in connecting the optical collection model with practical receiver characteristics, including field of view, detector sensitivity, bandwidth and noise. This could help extend the study towards communication-level performance and terminal design. Both directions would require further modelling and validation beyond the present work.</p>\n\n## Reproduction\n\n<p align="justify">Open the <a href="https://colab.research.google.com/github/Nishat-Yasmin-Anika/ground-to-air-fso-link-design/blob/main/notebooks/FSO_Link_Design_Colab.ipynb">standalone notebook in Google Colab</a> and select <strong>Runtime → Run all</strong>. The notebook runs on a CPU, generates its own source files, executes the tests and full experiments and exports the project. A Google Drive mount is not required.</p>\n\n<p align="center">\n  <a href="https://colab.research.google.com/github/Nishat-Yasmin-Anika/ground-to-air-fso-link-design/blob/main/notebooks/FSO_Link_Design_Colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Run the standalone notebook in Google Colab"></a>\n</p>\n\n<details>\n<summary>Local execution and regression comparison (Linux)</summary>\n\n```bash\npython -m pip install -r requirements.txt -r requirements-lock.txt\npython -m unittest discover -s tests -v\npython build_notebook.py --check\npython verify_manifest.py\n\nREF=$(mktemp -d)\ncp -r results/tables "$REF/tables"\nFAST_MODE=0 python run_study.py\npython compare_results.py "$REF/tables" results/tables\npython verify_manifest.py\n```\n\n<p align="justify">Run <code>python build_notebook.py</code> after editing any embedded source or documentation. Full mode uses 100,000 samples in the largest Monte Carlo runs. See the <a href="UPDATE_GITHUB.md">update instructions</a>.</p>\n\n</details>\n\n| Resource | Contents |\n|---|---|\n| [src/](src/) | Atmospheric and optical models |\n| [run_study.py](run_study.py) | Experiments and figure generation |\n| [tests/](tests/) | Mathematical and regression tests |\n| [compare_results.py](compare_results.py) | Numerical table comparison |\n| [notebooks/](notebooks/) | Standalone Colab notebook |\n| [results/](results/README.md) | Figures, tables and metadata |\n| [docs/](docs/) | Derivations, technical notes and source mapping |\n\n## References\n\n<ol>\n  <li><p align="justify">E. Erdogan et al., “Modeling Atmospheric Attenuation in Vertical FSO Links: An Altitude-Aware Piecewise Framework,” <em>IEEE Vehicular Technology Magazine</em>, 2026. <a href="https://doi.org/10.1109/MVT.2026.3687349">DOI</a> · <a href="https://arxiv.org/html/2604.18207v1">Author manuscript</a>.</p></li>\n  <li><p align="justify">M. Al Naboulsi, H. Sizun and F. de Fornel, “Fog attenuation prediction for optical and infrared waves,” <em>Optical Engineering</em>, vol. 43, pp. 319–329, 2004. See the <a href="docs/SOURCE_MAP.md">source map</a> for the coefficient relation, units and applicability.</p></li>\n  <li><p align="justify">A. A. Farid and S. Hranilovic, “Outage Capacity Optimization for Free-Space Optical Links with Pointing Errors,” 2007. <a href="https://doi.org/10.1109/JLT.2007.899174">DOI</a>.</p></li>\n  <li><p align="justify">M. Elamassie and M. Uysal, “Multi-Layer Airborne FSO Systems: Performance Analysis and Optimization,” 2025. <a href="https://doi.org/10.1109/TCOMM.2024.3471995">DOI</a>.</p></li>\n  <li><p align="justify">H. Rahimian and S. Mehrotra, “Frameworks and Results in Distributionally Robust Optimization,” 2022. <a href="https://doi.org/10.5802/ojmo.15">DOI</a>.</p></li>\n</ol>\n\n<p align="justify">The <a href="docs/SOURCE_MAP.md">source-to-implementation map</a> records the provenance of the coefficients and equations and identifies the additional assumptions made in this study.</p>\n\n## Citation and license\n\n<p align="justify">See <a href="CITATION.cff">CITATION.cff</a> for the software citation. The original scientific publications should be cited separately when their models are used. The code is distributed under the <a href="LICENSE">MIT License</a>.</p>\n\n<p align="justify">AI tools assisted implementation and review. The author is responsible for the scientific assumptions and findings. Suggestions and corrections are welcome.</p>\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `docs/SOURCE_MAP.md`

In [ ]:
p = PROJECT_ROOT / 'docs/SOURCE_MAP.md'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("# Source-to-implementation map\n\n## S1 — Layered atmosphere\nE. Erdogan, M. Elamassie, I. Altunbas, G. K. Kurt, M. Uysal and H. Yanikomeroglu, *Modeling Atmospheric Attenuation in Vertical FSO Links: An Altitude-Aware Piecewise Framework*, DOI [10.1109/MVT.2026.3687349](https://doi.org/10.1109/MVT.2026.3687349). Checked against the [arXiv author manuscript v1](https://arxiv.org/html/2604.18207v1), Sections III–IV and Tables I–II. The title of the manuscript differs from the journal title.\n\n`src/atmosphere.py` implements the printed Table II scenario boundaries and coefficients. For the rainy scenario, Section IV-A places the nimbostratus layer at 0.6–1.4 km; the remaining clear-air depth is split around it. Table II and Section IV-C differ in their lower clear-air layer; the default retains Table II for traceability, while `clear_prose` applies the 0–15 km clear-air description. The 1 km thick and 1 km light fog slabs reproduce Table II, not a typical measured fog depth. The source labels coefficients dB/km but uses the printed values in a natural exponential. The default follows literal power-decibel conversion, with the legacy branch as a separate interpretation. Original coefficient generation and MODTRAN comparisons were not reproduced. The published state probabilities are adopted conditionally; neither ERA5 nor a local weather data set has been processed. Independent states and five-category mixture are additional hypotheses.\n\n## S2 — Independent fog sensitivity model\nM. Al Naboulsi, H. Sizun and F. de Fornel, *Fog attenuation prediction for optical and infrared waves*, Optical Engineering 43 (2004), 319–329. The native radiation-fog extinction polynomial is `(0.18126 λ² + 0.13709 λ + 3.7502)/V` in km^-1, λ in µm, V in km. Conversion to power dB/km requires multiplying by 10/ln(10). This conversion and source range (0.69–1.55 µm, 0.05–1 km) are explicitly documented in [Validity-Aware Reliability Analysis and Power-Margin Design for Fog-Limited Free-Space Optical Links, Table 1](https://www.mdpi.com/2304-6732/13/11/1013) and the conversion is independently corroborated by [Nadeem et al., Radioengineering 2010](https://www.radioeng.cz/fulltexts/2010/10_04_596_600.pdf). A [2026 comparison article](https://doi.org/10.1038/s44459-026-00081-8) reports the polynomial directly in dB/km; the inconsistency is explicitly shown as a diagnostic in Table 10, not used as the native-unit model. The original 2004 paper's full text was not obtained. The 2026 validity-aware article also makes clear that the radiation relation is conditional on radiation fog; here it is a hypothetical stress case, not a claim that the source fog is radiation fog. Only the fog coefficients change, at 0.20 and 0.77 km visibility; slab depths and other coefficients remain Table II. No measured data or original fit reproduction is claimed. Do not combine these explicitly converted coefficients with the legacy exponential branch.\n\n## S3 — Gaussian aperture and pointing\nA. A. Farid and S. Hranilovic, [Outage Capacity Optimization for Free-Space Optical Links with Pointing Errors](https://doi.org/10.1109/JLT.2007.899174), 2007. Background for pointing effects; the exact noncentral chi-square aperture integral here is derived from normalized Gaussian irradiance, not their equivalent beamwidth approximation. See [SciPy ncx2](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ncx2.html). Bessel and independent Cartesian quadratures verify the aperture formula. Gaussian beam-quality relationship: [Gaussian Beam Optics technical guide](https://www.physics.utoronto.ca/apl/hene/Gaussian-Beam-Optics.pdf). The implied M² is a consistency check, not a measurement. An equivalent Gaussian radius is assumed even for M² > 1.\n\n## S4 — Airborne optical-link research context\nM. Elamassie and M. Uysal, [Multi-Layer Airborne FSO Systems: Performance Analysis and Optimization](https://doi.org/10.1109/TCOMM.2024.3471995), 2025. Conceptual motivation, not reproduction of its communication-system results. [Prof. Elamassie's publications](https://faculty.ozyegin.edu.tr/mohammedelamassie/publications/). No affiliation or unpublished materials claimed.\n\n## S5 — Probability ambiguity\nH. Rahimian and S. Mehrotra, [Frameworks and Results in Distributionally Robust Optimization](https://doi.org/10.5802/ojmo.15), 2022. Total variation is half L1 distance; the inner linear objective is minimized by mass transfer, checked against a separate HiGHS LP. Radius is a sensitivity parameter, not an estimated confidence set.\n\n## S6 — Engineering standards context\n[ITU-R P.1814-1](https://www.itu.int/rec/R-REC-P.1814-1-202509-I/en) (terrestrial FSO; background checklist, not a claim of compliance) and [ITU-R P.835-7](https://www.itu.int/rec/R-REC-P.835-7-202408-I/en) (reference atmospheres, not a source for the five-category prior). A full link budget requires calibrated atmospheric and device information including scintillation and receiver impairments.\n\n## Not literature-derived\n50 mW optical output, 70% optical efficiency, 2 mW threshold, 25 mm waist, 55 µrad divergence, per-axis jitter, 92% target and nominal prior (0.76,0.10,0.07,0.04,0.03) are illustrative. No practical receiver or service-level specification is represented. No explicit rain-rate or snow-rate law is implemented: rain means the source's nimbostratus mixture. All probabilities refer to snapshots under the specified model, not time-series uptime.\n", encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `docs/TECHNICAL_NOTE.md`

In [ ]:
p = PROJECT_ROOT / 'docs/TECHNICAL_NOTE.md'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("# Technical note — v4 conditional atmospheric model sensitivity\n\n## Theory\nFor flat horizontal atmospheric slabs with link height H and zenith z, path length is L=H/cos(z). Each intersected slab contributes its traversed depth divided by cos(z). For printed specific attenuation c_j in dB/km, the integrated attenuation Q=sum(d_j c_j) is in dB; the power ratio is T=10^(-Q/10). The exponential source-reproduction branch uses exp(-Q), equivalent to multiplying the printed coefficient by 10/ln(10) in dB. The distinction is not a correction to the original coefficient values. With independent categorical states, exact joint probabilities are products. E[T] and T(E[Q]) differ by Jensen's inequality; only the stochastic mean is used for mean-power calculations.\n\nNormalized Gaussian irradiance I(x,y)=2/(pi w²) exp[-2((x-r)²+y²)/w²]. A circular receiver radius a captures G=F_ncx2((2a/w)²; df=2, nc=(2r/w)²). Integrating the same irradiance in polar coordinates yields the Bessel formula; Cartesian slices provide an independent numerical check. The Gaussian-equivalent beam radius w=sqrt(w0²+(theta L)²) assumes waist at the transmitter and half-angle divergence. The implied M²=pi w0 theta/lambda is about 2.787 (not measured). For zero-mean independent Gaussian transverse offsets with per-axis SD s=L sigma_theta, R is Rayleigh. With P=Ptx eta T G, define A=P(P>=Pmin). Invert the monotone overlap to find the maximum admissible displacement; conditional success is 1-exp[-rcrit²/(2s²)] for s>0. For s=0 use the deterministic inequality. The zero-offset overlap is 1-exp(-2a²/w²). Averaging G over the pointing distribution gives 1-exp[-a²/(2(s²+w²/4))] by Gaussian convolution.\n\nWeather conditional success is the state-weighted average. For hypothetical prior q, minimize p·A over p>=0, sum p=1 and 0.5||p-q||1<=rho. Sorting and transferring mass from high to low reward solves this finite linear problem; a separately constructed LP checks it. Robust availability is nondecreasing in aperture because each conditional availability is nondecreasing. Therefore feasible-endpoint bisection (not continuous root finding) works even when zero-jitter availability is a step function. Radius is restricted to [0.025,0.50] m. The returned endpoint meets the target, with radius bracket <=1e-7 m unless already feasible at the lower bound.\n\nFor finite aperture, Gaussian capture is strictly below 1. A state with Ptx eta T <= Pmin cannot succeed even in perfect alignment. The limit as radius tends to infinity of finite-aperture success uses the strict inequality Ptx eta T > Pmin. Its robust aggregate is a necessary feasibility ceiling; equality with the target alone need not imply a finite feasible receiver. The legacy reference ceiling is 0.964-0.9rho for rho<=0.96; its 92% boundary is rho≈0.048889. This is not a universal threshold. The literal dB and separate fog model can give different ceilings.\n\n## What is tested\nTable II fixture values, layer intersections, exact state enumeration, Jensen inequality, closed-form Gaussian collection, Bessel and Cartesian quadrature, independent Cartesian+LP reconstruction of both reference designs, LP agreement for random priors, monotonicity and zero-jitter steps, exact threshold equality, slightly infeasible upper bounds, invalid numerical inputs, multiseed Monte Carlo success and mean-power checks. These are numerical verification only.\n\n## Interpretation\nThe main branch uses literal dB/km. Separate branches reproduce the disputed equation convention, apply the prose clear-weather layer, and stress-test fog coefficients with a published radiation-fog polynomial. Neither the fog stress case nor the original table is calibrated to local measurements. The code varies targets and priors, but the central physical assumptions remain hypothetical. See SOURCE_MAP.md for provenance and exclusions. Results are optical-power exceedance probabilities, not BER or communications uptime.\n", encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `docs/COMPLEXITY_AND_VERIFICATION.md`

In [ ]:
p = PROJECT_ROOT / 'docs/COMPLEXITY_AND_VERIFICATION.md'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('# Complexity and verification\n\nLet S be weather scenarios, K the maximum number of joint atmospheric slab states, L the number of layers, B inner inverse-overlap iterations and D outer aperture bisections. Enumeration O(SKL), conditional probability O(SKB) CDF evaluations, TV mass transfer O(S log S), aperture search O(D(SKB+S log S)). MC O(NL) random draws plus N numerical CDF evaluations, O(N) working memory. Exact analytic results do not depend on MC sample count.\n\nRun `python -m unittest discover -s tests -v` (44 automated mathematical, numerical and regression tests), `FAST_MODE=0 python run_study.py` and `python verify_manifest.py`. The new multi-case MC experiment uses three fixed seeds in four cases and compares both probability and analytic mean power using conservative five-standard-error diagnostic tolerances. Wilson intervals are individual sampling intervals, not model uncertainty. `results/metadata/manifest.json` contains SHA-256 fingerprints of the generating source and results (manifest itself excluded). A successful check shows integrity, not empirical validity. Figures are numerical outputs; SVG metadata and runtime can vary by environment. The notebook is built from canonical sources and checked by `python build_notebook.py --check`.\n\n\n## Numerical Regression Verification\n\nVersion 4.1 introduces automated comparison of regenerated\nCSV tables against previously committed reference tables.\n\nThe verification checks:\n\n- All 11 expected CSV filenames.\n- Column names and row counts.\n- Numerical values with relative tolerance 1e-8 and\n  absolute tolerance 1e-9.\n- Exact sample counts, random seeds and feasibility statuses.\n- Consistency of missing and NaN values.\n\nExecution timings are excluded from comparison.\n\nThe GitHub Actions workflow preserves the committed\nreference tables before running the full experiments\nwith FAST_MODE=0.\n\nAny unexpected numerical discrepancy causes the\nverification workflow to fail.\n\nThe reference scientific dependency versions are recorded\nin requirements-lock.txt. The environment that actually\ngenerated the committed results is recorded separately\nin results/metadata/environment.json.\n\nChanges in numerical libraries or random sampling\nimplementations may cause regression failures and must\nbe investigated before accepting new reference results.\n\nRegression agreement establishes computational\nreproducibility, not independent physical validation.\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `results/README.md`

In [ ]:
p = PROJECT_ROOT / 'results/README.md'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('# Simulation Results\n\nAll figures and numerical tables are generated automatically\nby `run_study.py`.\n\nThese results represent conditional computational predictions,\nnot experimental measurements.\n\n[Return to project overview](../README.md)\n\n## Figure Gallery\n\n### Independent Gaussian Aperture Integration\n\nComparison of analytical and independent numerical integration.\n\n![Independent Gaussian Aperture Integration](figures/01_aperture_integral_validation.png)\n\n[PNG](figures/01_aperture_integral_validation.png) |\n[Vector SVG](figures/01_aperture_integral_validation.svg)\n\n---\n\n### Atmospheric Coefficient Interpretations\n\nComparison of atmospheric attenuation conventions.\n\n![Atmospheric Coefficient Interpretations](figures/02_atmospheric_unit_interpretations.png)\n\n[PNG](figures/02_atmospheric_unit_interpretations.png) |\n[Vector SVG](figures/02_atmospheric_unit_interpretations.svg)\n\n---\n\n### Availability versus Aperture\n\nRelationship between aperture size and modeled availability.\n\n![Availability versus Aperture](figures/03_availability_vs_aperture.png)\n\n[PNG](figures/03_availability_vs_aperture.png) |\n[Vector SVG](figures/03_availability_vs_aperture.svg)\n\n---\n\n### Minimum Aperture Frontier\n\nReceiver aperture requirements under pointing uncertainty.\n\n![Minimum Aperture Frontier](figures/04_minimum_aperture_frontier.png)\n\n[PNG](figures/04_minimum_aperture_frontier.png) |\n[Vector SVG](figures/04_minimum_aperture_frontier.svg)\n\n---\n\n### Weather Probability Uncertainty\n\nSensitivity to alternative weather probability distributions.\n\n![Weather Probability Uncertainty](figures/05_weather_probability_shift.png)\n\n[PNG](figures/05_weather_probability_shift.png) |\n[Vector SVG](figures/05_weather_probability_shift.svg)\n\n---\n\n### Design Feasibility Map\n\nFeasibility across the evaluated design parameter space.\n\n![Design Feasibility Map](figures/06_design_feasibility_map.png)\n\n[PNG](figures/06_design_feasibility_map.png) |\n[Vector SVG](figures/06_design_feasibility_map.svg)\n\n---\n\n### Ablation Analysis\n\nIndividual contributions of selected channel-model components.\n\n![Ablation Analysis](figures/07_ablation.png)\n\n[PNG](figures/07_ablation.png) |\n[Vector SVG](figures/07_ablation.svg)\n\n---\n\n### Monte Carlo Verification\n\nStatistical comparison with analytical predictions.\n\n![Monte Carlo Verification](figures/08_monte_carlo_validation.png)\n\n[PNG](figures/08_monte_carlo_validation.png) |\n[Vector SVG](figures/08_monte_carlo_validation.svg)\n\n---\n\n### Atmospheric Feasibility Ceiling\n\nAtmosphere-limited upper bounds on modeled availability.\n\n![Atmospheric Feasibility Ceiling](figures/09_atmospheric_feasibility_ceiling.png)\n\n[PNG](figures/09_atmospheric_feasibility_ceiling.png) |\n[Vector SVG](figures/09_atmospheric_feasibility_ceiling.svg)\n\n---\n\n### Optimization Convention Sensitivity\n\nAperture requirements under alternative coefficient conventions.\n\n![Optimization Convention Sensitivity](figures/10_optimization_convention_sensitivity.png)\n\n[PNG](figures/10_optimization_convention_sensitivity.png) |\n[Vector SVG](figures/10_optimization_convention_sensitivity.svg)\n\n---\n\n### Atmospheric Model and Target Sensitivity\n\nComparison across atmospheric models and availability targets.\n\n![Atmospheric Model and Target Sensitivity](figures/11_model_and_target_sensitivity.png)\n\n[PNG](figures/11_model_and_target_sensitivity.png) |\n[Vector SVG](figures/11_model_and_target_sensitivity.svg)\n\n---\n\n### Fog Model Comparison\n\nConditional comparison of fog attenuation formulations.\n\n![Fog Model Comparison](figures/12_fog_model_benchmark.png)\n\n[PNG](figures/12_fog_model_benchmark.png) |\n[Vector SVG](figures/12_fog_model_benchmark.svg)\n\n---\n\n## Numerical Data\n\n- [01 Weather Scenarios And Conventions](tables/01_weather_scenarios_and_conventions.csv)\n- [02 Minimum Aperture Frontier](tables/02_minimum_aperture_frontier.csv)\n- [03 Tv Radius Sensitivity](tables/03_tv_radius_sensitivity.csv)\n- [04 Ablation](tables/04_ablation.csv)\n- [05 Monte Carlo Verification](tables/05_monte_carlo_verification.csv)\n- [06 Alternate Illustrative Weather Priors](tables/06_alternate_illustrative_weather_priors.csv)\n- [07 Atmospheric Feasibility Ceiling](tables/07_atmospheric_feasibility_ceiling.csv)\n- [08 Optimization Convention Sensitivity](tables/08_optimization_convention_sensitivity.csv)\n- [09 Model And Target Sensitivity](tables/09_model_and_target_sensitivity.csv)\n- [10 Fog Model Benchmark](tables/10_fog_model_benchmark.csv)\n- [11 Multi Case Mc](tables/11_multi_case_mc.csv)\n\n## Reproducibility Metadata\n\n- [Execution environment](metadata/environment.json)\n- [Experiment configuration](metadata/run_config.json)\n- [Runtime information](metadata/runtime.json)\n- [SHA-256 integrity manifest](metadata/manifest.json)\n\n## Scientific Interpretation\n\nThe primary atmospheric interpretation uses the literal\ndB/km conversion. Legacy exponential results are retained\nfor source traceability.\n\nThe atmospheric feasibility ceiling must be distinguished\nfrom infeasibility within a finite aperture search interval.\n\nMonte Carlo agreement supports internal numerical\nconsistency, not experimental validation.\n\nThe atmospheric coefficients, illustrative weather priors\nand hypothetical hardware parameters require further\nphysical investigation.\n\n[Technical note](../docs/TECHNICAL_NOTE.md) |\n[Scientific source map](../docs/SOURCE_MAP.md) |\n[Verification methodology](../docs/COMPLEXITY_AND_VERIFICATION.md)\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `requirements.txt`

In [ ]:
p = PROJECT_ROOT / 'requirements.txt'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('numpy>=1.24,<3\nscipy>=1.11,<2\nmatplotlib>=3.7,<4\nnbformat>=5.9,<6\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `requirements-lock.txt`

In [ ]:
p = PROJECT_ROOT / 'requirements-lock.txt'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('# Scientific dependency versions in the delivered local verification environment.\nnumpy==2.3.5\nscipy==1.17.0\nmatplotlib==3.10.8\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `verify_manifest.py`

In [ ]:
p = PROJECT_ROOT / 'verify_manifest.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Check exact bytes and SHA-256 of generated results and generating source."""\nimport json,hashlib\nfrom pathlib import Path\nroot=Path(__file__).resolve().parent\nout=root/\'results\'\nm=json.loads((out/\'metadata/manifest.json\').read_text())\nassert set(p.relative_to(out).as_posix() for p in out.rglob(\'*\') if p.is_file() and p.name!=\'manifest.json\' and not p.name.startswith(\'.\'))=={e[\'path\'] for e in m[\'files\']},\'Result inventory mismatch\'\nfor e in m[\'files\']:\n    b=(out/e[\'path\']).read_bytes()\n    assert len(b)==e[\'bytes\'] and hashlib.sha256(b).hexdigest()==e[\'sha256\'],e[\'path\']\nfor p,h in m[\'source_sha256\'].items():\n    assert hashlib.sha256((root/p).read_bytes()).hexdigest()==h,p\nprint(f"VERIFIED {len(m[\'files\'])} result files and {len(m[\'source_sha256\'])} generating source files")\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `build_notebook.py`

In [ ]:
p = PROJECT_ROOT / 'build_notebook.py'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Build the portable notebook from canonical repository files (no duplicated editing)."""\nimport argparse,hashlib,json\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parent\nDEST=ROOT/\'notebooks/FSO_Link_Design_Colab.ipynb\'\nFILES = [\'src/atmosphere.py\',\n \'src/fso_link.py\',\n \'run_study.py\',\n \'compare_results.py\',\n \'tests/test_link.py\',\n \'tests/test_audit.py\',\n \'README.md\',\n \'docs/SOURCE_MAP.md\',\n \'docs/TECHNICAL_NOTE.md\',\n \'docs/COMPLEXITY_AND_VERIFICATION.md\',\n \'results/README.md\',\n \'requirements.txt\',\n \'requirements-lock.txt\',\n \'verify_manifest.py\',\n \'build_notebook.py\',\n \'UPDATE_GITHUB.md\',\n \'.github/workflows/python-tests.yml\',\n \'docs/MATHEMATICAL_DERIVATIONS.md\',\n \'CITATION.cff\',\n \'LICENSE\',\n \'assets/cover.svg\',\n \'assets/method.svg\',\n \'.gitignore\']\n\ndef build():\n    cells=[]\n    def add(kind,source):\n        c={\'cell_type\':kind,\'id\':hashlib.sha256(f\'{len(cells)}:{kind}\'.encode()).hexdigest()[:12],\n           \'metadata\':{},\'source\':source.splitlines(keepends=True)}\n        if kind==\'code\':c.update(execution_count=None,outputs=[])\n        cells.append(c)\n    add(\'markdown\', \'\'\'# Atmospheric-model sensitivity in ground-to-air FSO receiver design — v4.1\n**Nishat Yasmin Anika · ECE · KUET**  \nStandalone CPU notebook generated from canonical source files. Run all cells in order; no Drive mount required.\n\n**Scope:** conditional optical-power availability, not BER, measured uptime or practical receiver specifications. Literal dB/km is the primary convention; equation-as-printed is a legacy comparison only. Table II, the clear-weather prose alternative and the hypothetical radiation-fog stress case are kept separate. No source MODTRAN data or field observations were reproduced. See the embedded source map.\n\nThe original repository is [ground-to-air-fso-link-design](https://github.com/Nishat-Yasmin-Anika/ground-to-air-fso-link-design). AI tools assisted code revision and review; author approval is required before publication.\'\'\')\n    add(\'markdown\',\'\'\'## 1. Mathematical model and verification\nFor integrated power attenuation $Q$ in dB, $T=10^{-Q/10}$. If the printed quantity instead represents Napierian optical depth, $T=e^{-Q}$. A circular aperture captures $G=F_{\\\\chi\'^2_2}((2a/w)^2;\\\\lambda=(2r/w)^2)$ from a displaced Gaussian spot. Independent equal-variance transverse Gaussian jitter implies Rayleigh radial displacement. Received power is $P=P_{tx}\\\\eta T G$; availability is $\\\\Pr(P\\\\ge P_{min})$. TV robustness minimizes the scenario-weighted success probability over a half-L1 probability ball. Feasible-endpoint bisection finds the minimum radius in the specified search interval, including threshold steps at zero jitter. All inputs are conditional assumptions. See the embedded technical note for derivations and limitations.\'\'\')\n    add(\'code\',\'\'\'import sys, subprocess, importlib.util\nfrom pathlib import Path\nmissing = [p for p in (\'numpy\',\'scipy\',\'matplotlib\') if importlib.util.find_spec(p) is None]\nif missing:\n    subprocess.check_call([sys.executable,\'-m\',\'pip\',\'install\',*missing])\nimport tempfile\nPROJECT_ROOT = Path(tempfile.mkdtemp(prefix=\'fso_v41_project_\', dir=Path.cwd()))\nprint(\'Project:\', PROJECT_ROOT)\'\'\')\n    add(\'markdown\',\'## 2. Canonical source files\\nThese cells are generated from the repository. To edit later, change the standalone source files and rebuild this notebook rather than maintaining two copies.\')\n    for file in FILES:\n        body=(ROOT/file).read_text(encoding=\'utf-8\')\n        add(\'markdown\',f\'### `{file}`\')\n        add(\'code\',f"p = PROJECT_ROOT / {file!r}\\np.parent.mkdir(parents=True, exist_ok=True)\\np.write_text({body!r}, encoding=\'utf-8\')\\nprint(\'Wrote\', p.relative_to(PROJECT_ROOT))")\n    add(\'markdown\',\'## 3. Run numerical regression tests\\nThe run stops if any test fails. These tests verify the model implementation, not its empirical accuracy.\')\n    add(\'code\', \'\'\'subprocess.run([sys.executable,\'-m\',\'unittest\',\'discover\',\'-s\',\'tests\',\'-v\'],cwd=PROJECT_ROOT,check=True)\'\'\')\n    add(\'markdown\',\'## 4. Run full experiments\\nFull mode uses 100,000 samples per largest MC run and each multi-case run. Set `FAST_MODE=True` for 20,000 samples. Analytic results are unchanged.\')\n    add(\'code\',\'\'\'import os\nFAST_MODE = False\nsubprocess.run([sys.executable,\'run_study.py\'],cwd=PROJECT_ROOT,env={**os.environ,\'FAST_MODE\':\'1\' if FAST_MODE else \'0\'},check=True)\nsubprocess.run([sys.executable,\'verify_manifest.py\'],cwd=PROJECT_ROOT,check=True)\'\'\')\n    add(\'markdown\',\'## 5. Inspect results\\nThe three figures below compare both source conventions, atmospheric models and fog coefficient assumptions. All twelve figures and eleven CSV tables are exported.\')\n    add(\'code\',\'\'\'from IPython.display import display, Image\nimport csv\nfor name in (\'10_optimization_convention_sensitivity\',\'11_model_and_target_sensitivity\',\'12_fog_model_benchmark\'):\n    display(Image(filename=str(PROJECT_ROOT/\'results\'/\'figures\'/(name+\'.png\'))))\nwith (PROJECT_ROOT/\'results/tables/09_model_and_target_sensitivity.csv\').open() as f:\n    for row in csv.DictReader(f):\n        if float(row[\'target\']) == .92:\n            print(row[\'profile\'], \'TV=\'+row[\'tv_radius\'], \'diameter_cm=\'+row[\'diameter_cm\'], \'ceiling=\'+row[\'finite_aperture_ceiling\'], row[\'status\'])\'\'\')\n    add(\'markdown\',\'## 6. Export GitHub-ready update\\nDownload `github_update.zip` from Colab Files. This archive uses repository-relative paths. Preserve the existing `.git` history. License, citation metadata and assets are included. Review `UPDATE_GITHUB.md` before uploading.\')\n    add(\'code\',\'\'\'from zipfile import ZipFile, ZIP_DEFLATED\nsubprocess.run([sys.executable,\'build_notebook.py\'],cwd=PROJECT_ROOT,check=True)\narchive=Path.cwd()/\'github_update.zip\'\nwith ZipFile(archive,\'w\',ZIP_DEFLATED,compresslevel=8) as z:\n    for p in sorted(PROJECT_ROOT.rglob(\'*\')):\n        if p.is_file() and \'__pycache__\' not in p.parts and not p.name.endswith(\'.pyc\'):\n            z.write(p,p.relative_to(PROJECT_ROOT).as_posix())\nprint(\'Ready:\',archive)\n# Optional in Google Colab:\n# from google.colab import files; files.download(str(archive))\'\'\')\n    return {\'cells\':cells,\'metadata\':{\'kernelspec\':{\'display_name\':\'Python 3\',\'language\':\'python\',\'name\':\'python3\'},\'language_info\':{\'name\':\'python\'}},\'nbformat\':4,\'nbformat_minor\':5}\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser();ap.add_argument(\'--check\',action=\'store_true\');args=ap.parse_args();nb=build()\n    if args.check:\n        existing=json.loads(DEST.read_text())\n        assert [(c[\'cell_type\'],c[\'source\']) for c in existing[\'cells\']]==[(c[\'cell_type\'],c[\'source\']) for c in nb[\'cells\']], \'Notebook source differs: regenerate with build_notebook.py\'\n        print(\'Notebook canonical sources verified\')\n    else:\n        DEST.parent.mkdir(parents=True,exist_ok=True);DEST.write_text(json.dumps(nb,indent=1,ensure_ascii=False)+\'\\n\',encoding=\'utf-8\');print(\'Built\',DEST)\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `UPDATE_GITHUB.md`

In [ ]:
p = PROJECT_ROOT / 'UPDATE_GITHUB.md'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('# Updating the FSO Research Repository\n\n## Version 4.1.0\n\nThis refinement preserves the v4 scientific model\nand numerical findings while improving validation,\ndocumentation and reproducibility.\n\n## Install dependencies\n\n    python -m pip install -r requirements.txt\n\nFor the pinned reference environment:\n\n    python -m pip install -r requirements.txt -r requirements-lock.txt\n\n## Run automated verification\n\n    python -m unittest discover -s tests -v\n    python verify_manifest.py\n    python build_notebook.py --check\n\n## Numerical regression verification\n\nPreserve reference tables before generating new results.\n\nThe following commands are for Linux or Google Colab:\n\n    REF=$(mktemp -d)\n    cp -r results/tables "$REF/tables"\n    FAST_MODE=0 python run_study.py\n    python compare_results.py "$REF/tables" results/tables\n    python verify_manifest.py\n\nInvestigate numerical discrepancies before accepting\nnew reference results.\n\n## Notebook regeneration\n\nAfter modifying canonical source or documentation:\n\n    python build_notebook.py\n    python build_notebook.py --check\n\nDo not edit the generated notebook independently.\n\n## GitHub update\n\nExtract the generated ZIP and upload its contents\nto the existing repository root.\n\nDo not upload the ZIP itself or an enclosing folder.\nPreserve the Git history and original attribution.\n\n## Scientific scope\n\nThe project demonstrates conditional optical-power\navailability and robust receiver aperture design.\n\nNo experimental calibration, communication-level\nBER validation or publication-level novelty is claimed.\n\nFurther physical modeling remains future research.\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `.github/workflows/python-tests.yml`

In [ ]:
p = PROJECT_ROOT / '.github/workflows/python-tests.yml'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('name: Numerical verification\n\non: [push, pull_request, workflow_dispatch]\n\npermissions:\n  contents: read\n\njobs:\n  numerical-tests:\n    runs-on: ubuntu-latest\n    timeout-minutes: 10\n\n    steps:\n      - uses: actions/checkout@v6\n\n      - uses: actions/setup-python@v6\n        with:\n          python-version: \'3.11\'\n\n      - name: Install dependencies\n        run: python -m pip install -r requirements.txt -r requirements-lock.txt\n\n      - name: Run mathematical and regression tests\n        run: python -m unittest discover -s tests -v\n\n      - name: Check notebook synchronization\n        run: python build_notebook.py --check\n\n      - name: Verify committed artifacts\n        run: python verify_manifest.py\n\n      - name: Preserve reference numerical results\n        run: cp -r results/tables "$RUNNER_TEMP/fso_reference_tables"\n\n      - name: Regenerate full experiments\n        run: python run_study.py\n        env:\n          FAST_MODE: \'0\'\n\n      - name: Compare regenerated numerical results\n        run: python compare_results.py "$RUNNER_TEMP/fso_reference_tables" results/tables\n\n      - name: Verify regenerated artifacts\n        run: python verify_manifest.py\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `docs/MATHEMATICAL_DERIVATIONS.md`

In [ ]:
p = PROJECT_ROOT / 'docs/MATHEMATICAL_DERIVATIONS.md'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("# Mathematical Derivations\n\n## Ground-to-Air Free-Space Optical Receiver Design\n\n**Author:** Nishat Yasmin Anika  \n**Department:** Electronics and Communication Engineering, KUET\n\nThis document derives the mathematical expressions implemented\nin `src/atmosphere.py` and `src/fso_link.py`.\n\nThe derivations establish internal mathematical consistency\nunder declared assumptions. They do not establish empirical\naccuracy or experimental validation.\n\n[Project overview](../README.md) |\n[Scientific source map](SOURCE_MAP.md) |\n[Technical note](TECHNICAL_NOTE.md)\n\n---\n\n## 1. Notation and Assumptions\n\n| Symbol | Definition | Unit |\n|---|---|---|\n| $H$ | Receiver altitude | km |\n| $\\theta$ | Zenith angle from local vertical | rad |\n| $L$ | Total slant-path length | m |\n| $d_j$ | Path length inside atmospheric layer j | km |\n| $c_{jk}$ | Printed atmospheric coefficient | convention-dependent |\n| $Q$ | Integrated printed attenuation coefficient | convention-dependent |\n| $T$ | Atmospheric power transmittance | dimensionless |\n| $\\lambda$ | Optical wavelength | m |\n| $w_0$ | Transmitter beam waist radius | m |\n| $w$ | Receiver-plane beam radius | m |\n| $\\Theta$ | Beam divergence half-angle | rad |\n| $a$ | Receiver aperture radius | m |\n| $r$ | Radial pointing displacement | m |\n| $\\sigma$ | Per-axis displacement standard deviation | m |\n| $G$ | Optical collection fraction | dimensionless |\n| $P_t$ | Transmitted optical power | mW |\n| $\\eta$ | Optical efficiency | dimensionless |\n| $P_{\\min}$ | Received-power threshold | mW |\n| $A_i$ | Conditional scenario availability | dimensionless |\n| $\\rho$ | Total-variation ambiguity radius | dimensionless |\n\n### Model assumptions\n\n- Flat Earth and horizontally stratified atmospheric layers.\n- Ground transmitter at zero reference altitude.\n- A circular receiver aperture normal to the nominal beam.\n- Isotropic Gaussian-equivalent receiver-plane intensity.\n- Independent zero-mean Gaussian pointing components.\n- Independent atmospheric layer states within each scenario.\n- Pointing displacement independent of atmospheric states.\n- Fixed transmitter power and optical efficiency.\n- No turbulence, scintillation, detector noise or BER model.\n\nAll weather priors and optical hardware parameters are\nillustrative unless explicitly identified as literature inputs.\n\n---\n\n## 2. Atmospheric Layer Geometry\n\nFor receiver altitude $H$ and zenith angle $\\theta$,\n\n$$\nL = \\frac{1000H}{\\cos\\theta}.\n$$\n\nThe factor 1000 converts kilometres to metres.\n\nFor atmospheric layer j extending from altitude $b_j$\nto altitude $t_j$, the intersected vertical thickness is\n\n$$\nh_j = \\max(0,\\min(H,t_j)-b_j).\n$$\n\nTherefore, the slant-path length within that layer is\n\n$$\nd_j = \\frac{h_j}{\\cos\\theta}.\n$$\n\nFor contiguous layers covering the entire link,\n\n$$\n\\sum_j d_j = \\frac{H}{\\cos\\theta}.\n$$\n\nThe implemented layer model is restricted to an altitude\nof at most 30 km and a zenith angle of at most 70 degrees.\n\nThis is a flat-Earth geometric approximation.\n\n**Implementation:** `path_lengths_km()` and `Link.length_m`.\n\n---\n\n## 3. Beer-Lambert Atmospheric Attenuation\n\nLet $\\alpha(s)$ denote a power extinction coefficient.\n\nThe local optical-power differential equation is\n\n$$\n\\frac{dP}{ds}=-\\alpha(s)P(s).\n$$\n\nDividing by optical power and integrating,\n\n$$\n\\int_{P_t}^{P_r}\\frac{dP}{P}\n=\n-\\int_0^L\\alpha(s)\\,ds.\n$$\n\nHence,\n\n$$\nT=\\frac{P_r}{P_t}\n=\n\\exp\\left(-\\int_0^L\\alpha(s)\\,ds\\right).\n$$\n\nFor piecewise constant extinction,\n\n$$\nT=\\exp\\left(-\\sum_j\\alpha_jd_j\\right).\n$$\n\nThe extinction coefficients and path lengths must use\ncompatible length units.\n\n### 3.1 Power-decibel conversion\n\nFor integrated attenuation $Q_{\\mathrm{dB}}$,\n\n$$\nQ_{\\mathrm{dB}}=-10\\log_{10}T.\n$$\n\nRearranging,\n\n$$\nT=10^{-Q_{\\mathrm{dB}}/10}.\n$$\n\nConsequently, the relationship between power attenuation\ncoefficient in dB/km and extinction in inverse kilometres is\n\n$$\n\\alpha=\\frac{\\ln 10}{10}\\gamma.\n$$\n\n### 3.2 Implemented coefficient interpretations\n\nDefine the integrated printed coefficient as\n\n$$\nQ=\\sum_j c_jd_j.\n$$\n\nThe primary interpretation is\n\n$$\nT_{\\mathrm{dB}}=10^{-Q/10}.\n$$\n\nThe legacy interpretation is\n\n$$\nT_{\\mathrm{legacy}}=e^{-Q}.\n$$\n\nTheir corresponding attenuation values in dB are\n\n$$\n\\mathcal L_{\\mathrm{dB}}=Q\n$$\n\nand\n\n$$\n\\mathcal L_{\\mathrm{legacy}}\n=\\frac{10Q}{\\ln 10}.\n$$\n\nThese interpretations are kept separate because the source\nmanuscript's printed units and exponential convention\nrequire further reconciliation.\n\nThe legacy branch reproduces the equation-as-printed\ninterpretation, not the source's original MODTRAN validation.\n\n**Implementation:** `transmittance_from_q()`,\n`loss_db_from_q()`.\n\n**Reference:** S1 in `SOURCE_MAP.md`.\n\n---\n\n## 4. Atmospheric State Probabilities\n\nSuppose layer j has categorical states k with probabilities\n$p_{jk}$ and printed coefficients $c_{jk}$.\n\nThe probabilities satisfy\n\n$$\n\\sum_k p_{jk}=1.\n$$\n\nFor one selected state in every active layer,\n\n$$\nQ_{\\boldsymbol{k}}=\\sum_jd_jc_{j,k_j}.\n$$\n\nAssuming conditional independence across layers,\n\n$$\np_{\\boldsymbol{k}}=\\prod_jp_{j,k_j}.\n$$\n\nThe expected atmospheric transmittance is therefore\n\n$$\n\\mathbb E[T]\n=\n\\sum_{\\boldsymbol{k}}\np_{\\boldsymbol{k}}f(Q_{\\boldsymbol{k}}),\n$$\n\nwhere $f(Q)$ is the selected attenuation convention.\n\n### 4.1 Effective versus expected transmittance\n\nThe effective transmittance is\n\n$$\nT_{\\mathrm{eff}}=f(\\mathbb E[Q]).\n$$\n\nIn general,\n\n$$\nf(\\mathbb E[Q])\\ne\\mathbb E[f(Q)].\n$$\n\nBoth exponential attenuation mappings are convex in Q.\n\nBy Jensen's inequality,\n\n$$\n\\mathbb E[f(Q)]\\geq f(\\mathbb E[Q]).\n$$\n\nThus the transmittance of the mean coefficient must not\nbe confused with mean transmittance.\n\n### 4.2 Factorized expectation\n\nFor either convention,\n\n$$\nf(Q)=e^{-\\kappa Q},\n$$\n\nwhere $\\kappa=1$ for the legacy convention and\n$\\kappa=\\ln(10)/10$ for literal dB conversion.\n\nUnder layer independence,\n\n$$\n\\mathbb E[T]\n=\n\\prod_j\\left[\n\\sum_k p_{jk}e^{-\\kappa d_jc_{jk}}\n\\right].\n$$\n\nThis expression avoids explicit enumeration when only\nmean atmospheric transmittance is required.\n\nAn atmospheric loss-budget exceedance probability is\n\n$$\n\\Pr(\\mathcal L>B)\n=\n\\sum_{\\boldsymbol{k}}p_{\\boldsymbol{k}}\n\\mathbf 1\\{\\mathcal L_{\\boldsymbol{k}}>B\\}.\n$$\n\nThis quantity is not receiver outage probability.\n\n**Implementation:** `exact_distribution()`,\n`exact_statistics()`, `effective_transmittance()`,\n`mean_transmittance_factorized()`.\n\n---\n\n## 5. Gaussian Beam Propagation\n\nThe implemented Gaussian-equivalent beam radius is\n\n$$\nw(L)=\\sqrt{w_0^2+(\\Theta L)^2}.\n$$\n\nThe radius w describes the 1/e-squared intensity radius.\n\nFor a beam originating at its waist, the Gaussian-equivalent\nbeam-quality relationship is\n\n$$\n\\Theta=\\frac{M^2\\lambda}{\\pi w_0}.\n$$\n\nConsequently,\n\n$$\nM^2=\\frac{\\pi w_0\\Theta}{\\lambda}.\n$$\n\nThe existing illustrative parameters are\n\n- Wavelength: 1550 nm.\n- Beam waist radius: 25 mm.\n- Divergence half-angle: 55 microradians.\n\nThese imply a Gaussian-equivalent beam propagation factor\nof approximately 2.79.\n\nThis factor is not a measured beam-quality value.\nA nonideal beam need not possess a perfectly Gaussian\nintensity profile. The Gaussian-equivalent profile\nis an explicit simplifying assumption.\n\n**Implementation:** `Link.beam_radius_m`,\n`Link.implied_m_squared()` (property).\n\n**Reference:** S3 in `SOURCE_MAP.md`.\n\n---\n\n## 6. Exact Circular-Aperture Collection\n\nThe normalized displaced Gaussian intensity is\n\n$$\nI(x,y)=\\frac{2}{\\pi w^2}\n\\exp\\left[-\\frac{2((x-r)^2+y^2)}{w^2}\\right].\n$$\n\nIts integral over the entire transverse plane is unity:\n\n$$\n\\int_{\\mathbb R^2}I(x,y)\\,dx\\,dy=1.\n$$\n\nThe fraction captured by a circular aperture of radius a is\n\n$$\nG(a,w,r)\n=\n\\iint_{x^2+y^2\\leq a^2}I(x,y)\\,dx\\,dy.\n$$\n\n### 6.1 Aligned Gaussian beam\n\nFor zero displacement, radial integration gives\n\n$$\nG(a,w,0)\n=\n\\int_0^a\n\\frac{4s}{w^2}e^{-2s^2/w^2}\\,ds.\n$$\n\nHence,\n\n$$\nG(a,w,0)=1-e^{-2a^2/w^2}.\n$$\n\n### 6.2 Noncentral chi-square derivation\n\nThe normalized intensity corresponds to independent\nGaussian transverse coordinates:\n\n$$\nX\\sim\\mathcal N(r,w^2/4),\n\\qquad\nY\\sim\\mathcal N(0,w^2/4).\n$$\n\nThen\n\n$$\n\\frac{4(X^2+Y^2)}{w^2}\n\\sim\n\\chi'^2_2\\left(\\frac{4r^2}{w^2}\\right).\n$$\n\nTherefore,\n\n$$\nG(a,w,r)\n=\nF_{\\chi'^2_2}\n\\left(\n\\frac{4a^2}{w^2};\n\\frac{4r^2}{w^2}\n\\right).\n$$\n\nThis is exact for the assumed circular aperture and\nnormalized isotropic Gaussian intensity.\n\nIt is not an approximate equivalent-beamwidth formula.\n\n**Implementation:** `gaussian_aperture_capture()`.\n\n---\n\n## 7. Independent Bessel Integration\n\nUsing polar coordinates about the aperture center,\n\n$$\n(x-r)^2+y^2=s^2+r^2-2sr\\cos\\phi.\n$$\n\nSubstituting into the two-dimensional integral,\n\n$$\nG=\n\\frac{2}{\\pi w^2}\n\\int_0^a s e^{-2(s^2+r^2)/w^2}\n\\int_0^{2\\pi}e^{4sr\\cos\\phi/w^2}\n\\,d\\phi\\,ds.\n$$\n\nUsing the modified Bessel identity\n\n$$\n\\int_0^{2\\pi}e^{z\\cos\\phi}d\\phi\n=2\\pi I_0(z),\n$$\n\nwe obtain\n\n$$\nG=\n\\int_0^a\n\\frac{4s}{w^2}\ne^{-2(s^2+r^2)/w^2}\nI_0\\left(\\frac{4sr}{w^2}\\right)ds.\n$$\n\nFor numerical stability, define\n\n$$\nI_{0e}(z)=e^{-z}I_0(z),\\qquad z\\geq0.\n$$\n\nThe integrand becomes\n\n$$\nG=\n\\int_0^a\n\\frac{4s}{w^2}\ne^{-2(s-r)^2/w^2}\nI_{0e}\\left(\\frac{4sr}{w^2}\\right)ds.\n$$\n\nThis provides an independent integration method for\nverifying the noncentral chi-square calculation.\n\nThe test suite also uses Cartesian quadrature,\nintegrating transverse Gaussian slices independently.\n\n**Implementation:** `capture_via_independent_quadrature()`.\n\n---\n\n## 8. Pointing Displacement\n\nLet the horizontal and vertical pointing displacement\ncomponents be independent Gaussian variables:\n\n$$\nX_p,Y_p\\sim\\mathcal N(0,\\sigma^2).\n$$\n\nUnder the small-angle displacement approximation,\n\n$$\n\\sigma=L\\sigma_\\theta,\n$$\n\nwhere $\\sigma_\\theta$ is the per-axis angular\nstandard deviation in radians.\n\nThe radial displacement is\n\n$$\nR=\\sqrt{X_p^2+Y_p^2}.\n$$\n\nThe joint density is\n\n$$\nf(x,y)=\\frac{1}{2\\pi\\sigma^2}\ne^{-(x^2+y^2)/(2\\sigma^2)}.\n$$\n\nTransforming to polar coordinates and integrating over angle,\n\n$$\nf_R(r)=\n\\frac{r}{\\sigma^2}\ne^{-r^2/(2\\sigma^2)},\\qquad r\\geq0.\n$$\n\nThus R follows a Rayleigh distribution with CDF\n\n$$\nF_R(r)=1-e^{-r^2/(2\\sigma^2)}.\n$$\n\nWhen $\\sigma=0$, the displacement is deterministically zero.\nThat case must be handled separately.\n\nThis model excludes boresight bias, anisotropic jitter,\nangle-of-arrival loss and temporal tracking dynamics.\n\n---\n\n## 9. Mean Gaussian Collection\n\nThe Gaussian optical intensity has per-axis variance\n\n$$\n\\sigma_b^2=\\frac{w^2}{4}.\n$$\n\nIndependent Gaussian pointing displacement adds\nanother variance $\\sigma^2$.\n\nThe convolution therefore produces a centered Gaussian with\n\n$$\n\\sigma_{\\mathrm{total}}^2=\\frac{w^2}{4}+\\sigma^2.\n$$\n\nIntegrating over the circular aperture,\n\n$$\n\\mathbb E[G]\n=\n1-\\exp\\left[\n-\\frac{a^2}{2(\\sigma^2+w^2/4)}\n\\right].\n$$\n\nWhen pointing jitter approaches zero, this reduces to\nthe aligned Gaussian collection expression.\n\n**Implementation:** `mean_capture_analytic()`.\n\n---\n\n## 10. Received Optical Power\n\nThe link model combines transmitter power,\nfixed optical efficiency, atmospheric transmission\nand Gaussian aperture collection:\n\n$$\nP_r=P_t\\eta T G(a,w,R).\n$$\n\nThe corresponding power-loss budget in dB is\n\n$$\n\\mathcal L_{\\mathrm{total}}\n=\n-10\\log_{10}\\eta\n-10\\log_{10}T\n-10\\log_{10}G.\n$$\n\nThe expression is defined for strictly positive factors.\n\nUnder atmospheric and pointing independence,\n\n$$\n\\mathbb E[P_r]\n=\nP_t\\eta\\mathbb E[T]\\mathbb E[G].\n$$\n\nThe calculation describes collected optical power,\nnot electrical SNR or communication BER.\n\n---\n\n## 11. Conditional Received-Power Availability\n\nFor a fixed atmospheric transmittance t, define the\nrequired collection fraction as\n\n$$\ng_{\\min}=\\frac{P_{\\min}}{P_t\\eta t}.\n$$\n\nIf t is zero or the required collection fraction is\nat least unity, finite-aperture reception cannot\nsatisfy the threshold.\n\nFor a circular aperture and an isotropic Gaussian,\ncollection is nonincreasing with radial displacement.\n\nDefine the critical displacement by\n\n$$\nG(a,w,r_{\\mathrm{crit}})=g_{\\min}.\n$$\n\nFor nonzero jitter and an achievable collection requirement,\n\n$$\nA(t,a)=\n\\Pr(R\\leq r_{\\mathrm{crit}})\n=\n1-\\exp\\left[\n-\\frac{r_{\\mathrm{crit}}^2}{2\\sigma^2}\n\\right].\n$$\n\nIf the required collection equals the aligned maximum,\nonly zero displacement succeeds. This event has zero\nprobability under continuous, nonzero Rayleigh jitter.\n\nFor deterministic zero jitter,\n\n$$\nA(t,a)\n=\n\\mathbf 1\\{\nP_t\\eta tG(a,w,0)\\geq P_{\\min}\n\\}.\n$$\n\nThe critical displacement is calculated numerically\nusing a bracketed scalar root finder.\n\n**Implementation:** `critical_offset()`,\n`success_probability_for_transmittance()`.\n\n---\n\n## 12. Weather-Mixture Availability\n\nFor weather scenario i, enumerate its atmospheric states\nwith conditional probabilities $p_{ik}$.\n\nIts conditional availability is\n\n$$\nA_i(a)=\\sum_kp_{ik}A(t_{ik},a).\n$$\n\nFor an illustrative scenario probability vector\n\n$$\n\\boldsymbol p=(0.76,0.10,0.07,0.04,0.03),\n$$\n\nthe nominal availability is\n\n$$\nA_{\\mathrm{nom}}(a)\n=\n\\sum_{i=1}^{5}p_iA_i(a).\n$$\n\nThe scenario ordering is clear, pollution, fog, rain and snow.\n\nThese outer scenario probabilities are hypothetical.\nThey are not estimated from ERA5 or historical weather data.\n\nAvailability here denotes a modeled snapshot probability,\nnot measured time-series uptime.\n\n**Implementation:** `conditional_availability()`,\n`availability()`.\n\n---\n\n## 13. Total-Variation Robust Optimization\n\nThe actual scenario probability vector q is assumed\nto belong to the ambiguity set\n\n$$\n\\mathcal U_\\rho=\n\\left\\{\n\\boldsymbol q:\nq_i\\geq0,\\;\n\\sum_iq_i=1,\\;\n\\frac12\\sum_i|q_i-p_i|\\leq\\rho\n\\right\\}.\n$$\n\nThe robust availability is\n\n$$\nA_{\\mathrm{rob}}(a)\n=\n\\min_{\\boldsymbol q\\in\\mathcal U_\\rho}\n\\sum_iq_iA_i(a).\n$$\n\n### 13.1 Greedy probability transfer\n\nTransferring probability mass delta from scenario j\nto scenario i changes the objective by\n\n$$\n\\Delta A=\\delta(A_i-A_j).\n$$\n\nIf $A_i<A_j$, the transfer reduces availability.\n\nThe greatest reduction per unit of transferred mass\nis obtained by moving mass from the highest-availability\nremaining donor to the lowest-availability available recipient.\n\nThe algorithm respects\n\n- Available donor probability.\n- Recipient probability capacity.\n- Remaining total-variation budget.\n\nSorting and successive probability transfers solve the\nfinite-state linear objective, up to numerical tolerances.\n\n### 13.2 Independent linear program\n\nIntroduce auxiliary nonnegative variables $u_i$ satisfying\n\n$$\nq_i-p_i\\leq u_i,\n\\qquad\np_i-q_i\\leq u_i.\n$$\n\nThen solve\n\n$$\n\\min_{\\boldsymbol q,\\boldsymbol u}\n\\sum_iq_iA_i\n$$\n\nsubject to\n\n$$\n\\sum_iq_i=1,\\qquad q_i\\geq0,\n$$\n\n$$\nu_i\\geq0,\\qquad \\sum_i u_i\\leq2\\rho.\n$$\n\nThe implemented LP independently verifies the\ngreedy probability-transfer solution.\n\nThe ambiguity radius represents an assumed uncertainty\nbudget, not an empirically established confidence region.\n\n**Implementation:** `worst_case_tv_greedy()`,\n`worst_case_tv_lp()`.\n\n**Reference:** S5 in `SOURCE_MAP.md`.\n\n---\n\n## 14. Minimum Receiver-Aperture Optimization\n\nThe design objective is\n\n$$\n\\min_a a\n$$\n\nsubject to\n\n$$\nA_{\\mathrm{rob}}(a)\\geq A_{\\mathrm{target}}.\n$$\n\nThe implemented search interval is\n\n$$\n0.025\\leq a\\leq0.50\\ \\mathrm{m}.\n$$\n\nFor fixed beam width and displacement, increasing the\ncircular aperture cannot reduce collected optical power,\nbecause the collection region only increases.\n\nTherefore, conditional, nominal and robust availability\nare all nondecreasing with aperture radius.\n\nThis permits monotone bisection.\n\nAt each iteration,\n\n$$\na_{\\mathrm{mid}}\n=\n\\frac{a_{\\mathrm{lo}}+a_{\\mathrm{hi}}}{2}.\n$$\n\nIf the midpoint satisfies the target, it becomes\nthe feasible upper endpoint. Otherwise, it becomes\nthe infeasible lower endpoint.\n\nThe algorithm returns the feasible upper endpoint\nwith a radius tolerance of $10^{-7}$ m.\n\nThis formulation also handles discontinuous zero-jitter\navailability steps.\n\nIf the lower bound already satisfies the requirement,\nthe lower bound is returned.\n\nIf the upper bound fails, no feasible aperture exists\ninside the specified interval.\n\nThe reported receiver diameter is\n\n$$\nD=2a.\n$$\n\n**Implementation:** `min_aperture_for_target()`.\n\n---\n\n## 15. Atmospheric Feasibility Ceiling\n\nFor every finite aperture and finite Gaussian beam width,\n\n$$\nG(a,w,r)<1.\n$$\n\nConsequently, a necessary condition for successful\nfinite-aperture reception is\n\n$$\nP_t\\eta t>P_{\\min}.\n$$\n\nThe inequality is strict because a finite Gaussian\naperture cannot capture exactly all transmitted power.\n\nThe scenario-level limiting availability is\n\n$$\nC_i=\n\\sum_kp_{ik}\n\\mathbf 1\\{P_t\\eta t_{ik}>P_{\\min}\\}.\n$$\n\nThe nominal atmospheric ceiling is\n\n$$\nC_{\\mathrm{nom}}=\\sum_ip_iC_i.\n$$\n\nThe robust atmospheric ceiling is\n\n$$\nC_{\\mathrm{rob}}\n=\n\\min_{\\boldsymbol q\\in\\mathcal U_\\rho}\n\\sum_iq_iC_i.\n$$\n\nIf\n\n$$\nC_{\\mathrm{rob}}<A_{\\mathrm{target}},\n$$\n\nno finite aperture can satisfy the robust target.\n\nEquality between the ceiling and target does not itself\nestablish finite-aperture feasibility.\n\nThe critical TV radius identifies the break-even\nboundary of the atmospheric ceiling. It must not be\ninterpreted as a guarantee of finite-aperture feasibility.\n\n**Implementation:** `weather_limited_ceiling()`,\n`critical_tv_radius()`.\n\n---\n\n## 16. Monte Carlo Verification\n\nFor every simulated realization:\n\n1. Draw a weather scenario.\n2. Draw categorical atmospheric layer states.\n3. Calculate atmospheric transmittance.\n4. Draw two Gaussian pointing components.\n5. Calculate radial displacement.\n6. Evaluate Gaussian aperture collection.\n7. Calculate received optical power.\n8. Determine whether the power threshold is satisfied.\n\nFor N independent draws with K successes,\n\n$$\n\\widehat A=\\frac{K}{N}.\n$$\n\nIts binomial variance under the assumed simulation model is\n\n$$\n\\operatorname{Var}(\\widehat A)\n=\n\\frac{A(1-A)}{N}.\n$$\n\n### 16.1 Wilson confidence interval\n\nThe Wilson interval is obtained by inverting the\nbinomial normal-score inequality:\n\n$$\n(\\widehat A-A)^2\n\\leq\n\\frac{z^2 A(1-A)}{N}.\n$$\n\nSolving the resulting quadratic inequality gives\nthe interval center\n\n$$\nc=\n\\frac{\\widehat A+z^2/(2N)}\n{1+z^2/N}\n$$\n\nand half-width\n\n$$\nh=\n\\frac{z}{1+z^2/N}\n\\sqrt{\n\\frac{\\widehat A(1-\\widehat A)}{N}\n+\n\\frac{z^2}{4N^2}\n}.\n$$\n\nThe confidence interval is\n\n$$\n[c-h,\\;c+h].\n$$\n\nFor the approximate 95 percent interval,\n\n$$\nz\\approx1.959964.\n$$\n\nMonte Carlo agreement supports the probability\nimplementation under the simulated model.\n\nIt does not validate the atmospheric assumptions,\nhardware performance or real-world availability.\n\n**Implementation:** `sample_link()`, `wilson_score()`.\n\n---\n\n## 17. Reference Numerical Example\n\nConsider the existing illustrative link:\n\n| Parameter | Value |\n|---|---:|\n| Altitude | 3 km |\n| Zenith angle | 0 degrees |\n| Wavelength | 1550 nm |\n| Transmitter power | 50 mW |\n| Optical efficiency | 0.70 |\n| Waist radius | 0.025 m |\n| Divergence half-angle | 55 microradians |\n| Per-axis angular jitter | 15 microradians |\n| Receiver power threshold | 2 mW |\n| Availability target | 0.92 |\n\nThe link length is\n\n$$\nL=3000\\ \\mathrm{m}.\n$$\n\nThe Gaussian-equivalent beam radius is\n\n$$\nw=\\sqrt{0.025^2+(55\\times10^{-6}\\times3000)^2}\n\\approx0.166883\\ \\mathrm{m}.\n$$\n\nThe per-axis pointing displacement deviation is\n\n$$\n\\sigma=3000\\times15\\times10^{-6}\n=0.045\\ \\mathrm{m}.\n$$\n\nThe available optical power before atmospheric and\naperture losses is\n\n$$\nP_t\\eta=35\\ \\mathrm{mW}.\n$$\n\nFor the primary clear-weather Table II scenario\nat 3 km altitude,\n\n$$\nQ=3(0.034)=0.102.\n$$\n\nThe two atmospheric interpretations give\n\n$$\nT_{\\mathrm{dB}}=10^{-0.102/10}\\approx0.976787\n$$\n\nand\n\n$$\nT_{\\mathrm{legacy}}=e^{-0.102}\\approx0.903030.\n$$\n\nThe verified scenario-mixture optimization results are:\n\n| Atmospheric interpretation | Nominal diameter | TV radius 0.04 |\n|---|---:|---:|\n| Literal dB/km | 9.472 cm | 11.816 cm |\n| Legacy exponential | 21.947 cm | 27.009 cm |\n\nThe nominal and robust results combine all five\nweather scenarios, not only the clear-weather example.\n\nThese dimensions are conditional numerical predictions.\n\nSee the complete generated results in\n[the results directory](../results/README.md).\n\n---\n\n## 18. Computational Complexity and Limitations\n\nLet J be the number of active atmospheric layers\nand $K_j$ the number of states within layer j.\n\nExplicit enumeration requires\n\n$$\nN_{\\mathrm{states}}=\\prod_{j=1}^J K_j.\n$$\n\nIts time complexity is approximately\n\n$$\nO\\left(J\\prod_{j=1}^J K_j\\right).\n$$\n\nThe factorized mean-transmittance calculation requires\n\n$$\nO\\left(\\sum_jK_j\\right).\n$$\n\nFor S weather scenarios, the greedy TV solver\nrequires sorting and probability transfer:\n\n$$\nO(S\\log S).\n$$\n\nFor aperture interval width W and absolute tolerance\nepsilon, bisection requires approximately\n\n$$\nO\\left(\\log_2\\frac{W}{\\epsilon}\\right)\n$$\n\navailability evaluations.\n\nActual runtime also depends on atmospheric enumeration,\nnoncentral chi-square evaluations and scalar root finding.\n\n### Scientific limitations\n\n- Atmospheric coefficient calibration remains unresolved.\n- The two attenuation conventions are sensitivity cases.\n- Weather frequencies and TV radii are illustrative.\n- Layer-state independence is an assumption.\n- The Gaussian-equivalent beam is not measured.\n- Atmospheric turbulence and scintillation are excluded.\n- Receiver noise, modulation and BER are excluded.\n- The radiation-fog sensitivity model has conditional\n  applicability; its original source requires further review.\n- No experimental optical hardware validation is claimed.\n\nThe mathematical and numerical verification establishes\nconsistency of the implemented model, not universal\nphysical validity.\n\n## References\n\nThe equation derivations use standard probability,\nGaussian optics, attenuation and optimization theory.\n\nFor original scientific references, coefficient provenance\nand applicability qualifications, consult\n[SOURCE_MAP.md](SOURCE_MAP.md).\n\nKey references include:\n\n1. Erdogan et al., DOI: 10.1109/MVT.2026.3687349.\n2. Farid and Hranilovic, DOI: 10.1109/JLT.2007.899174.\n3. Elamassie and Uysal, DOI: 10.1109/TCOMM.2024.3471995.\n4. Rahimian and Mehrotra, DOI: 10.5802/ojmo.15.\n5. ITU-R P.1814-1 and P.835-7.\n\nThese references provide theoretical and research context.\nTheir complete system results are not claimed to have\nbeen independently reproduced.\n", encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `CITATION.cff`

In [ ]:
p = PROJECT_ROOT / 'CITATION.cff'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('cff-version: 1.2.0\nmessage: "If you use this software, please cite this repository and the relevant scientific references separately."\n\ntitle: "Atmospheric-Model Sensitivity in Ground-to-Air FSO Receiver Design"\n\ntype: software\nversion: 4.1.0\n\nauthors:\n  - family-names: "Anika"\n    given-names: "Nishat Yasmin"\n    affiliation: "Khulna University of Engineering & Technology"\n\nrepository-code: "https://github.com/Nishat-Yasmin-Anika/ground-to-air-fso-link-design"\n\nlicense: MIT\ndate-released: 2026-11-28\n\nabstract: >\n  A reproducible computational study investigating ground-to-air\n  free-space optical receiver aperture requirements under\n  atmospheric attenuation, Gaussian beam collection, pointing\n  displacement and weather probability uncertainty.\n  The framework incorporates total-variation distributionally\n  robust optimization and independent numerical verification.\n  The results represent conditional model predictions rather\n  than experimentally validated communication performance.\n\nkeywords:\n  - free-space optical communication\n  - ground-to-air optical links\n  - atmospheric attenuation\n  - Gaussian beam propagation\n  - optical receiver aperture\n  - pointing errors\n  - distributionally robust optimization\n  - scientific computing\n  - numerical verification\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `LICENSE`

In [ ]:
p = PROJECT_ROOT / 'LICENSE'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('MIT License\n\nCopyright (c) 2026 Nishat Yasmin Anika\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `assets/cover.svg`

In [ ]:
p = PROJECT_ROOT / 'assets/cover.svg'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1440 544" role="img" aria-labelledby="heading subheading">\n<title id="heading">Free-space optical link design research cover</title><desc id="subheading">A laser link passing between atmospheric layers and a high-altitude aerial platform, with an abstract receiver pattern.</desc>\n<defs>\n  <linearGradient id="bg" x1="0" y1="0" x2="1" y2="1"><stop offset="0" stop-color="#071A2C"/><stop offset=".64" stop-color="#0A253B"/><stop offset="1" stop-color="#143C4F"/></linearGradient>\n  <linearGradient id="beam" x1="0" y1="1" x2="0" y2="0"><stop offset="0" stop-color="#F6D78B"/><stop offset=".56" stop-color="#5DE7E0"/><stop offset="1" stop-color="#D2FFFF"/></linearGradient>\n  <radialGradient id="halo"><stop offset="0" stop-color="#70E2E3" stop-opacity=".23"/><stop offset="1" stop-color="#70E2E3" stop-opacity="0"/></radialGradient>\n  <pattern id="grid" width="36" height="36" patternUnits="userSpaceOnUse"><path d="M36 0H0V36" fill="none" stroke="#86B6C9" stroke-width=".8" opacity=".07"/></pattern>\n  <filter id="soft"><feGaussianBlur stdDeviation="11"/></filter>\n</defs>\n<rect width="1440" height="544" rx="23" fill="url(#bg)"/>\n<rect width="1440" height="544" rx="23" fill="url(#grid)"/>\n<g opacity=".28" stroke="#9CD0DA" stroke-dasharray="3 10"><path d="M765 110H1400"/><path d="M765 205H1400"/><path d="M765 300H1400"/><path d="M765 393H1400"/></g>\n<g font-family="Arial, Helvetica, sans-serif">\n <rect x="75" y="75" width="230" height="31" rx="15.5" fill="#153D50" stroke="#39778A" stroke-width="1"/>\n <circle cx="96" cy="90" r="4" fill="#65E7D9"/><text x="111" y="95" fill="#BFDFE5" font-size="12" font-weight="700" letter-spacing="2">NUMERICAL STUDY</text>\n <text x="75" y="173" fill="#F8FCFD" font-size="55" font-weight="700" letter-spacing="-.8">Ground-to-air</text>\n <text x="75" y="231" fill="#F8FCFD" font-size="55" font-weight="700" letter-spacing="-.8">optical links</text>\n <text x="75" y="277" fill="#70DCD7" font-size="22" letter-spacing=".2">Designing for uncertainty</text>\n <text x="75" y="333" fill="#A9CBD4" font-size="18">Atmospheric layers · pointing jitter</text>\n <text x="75" y="361" fill="#A9CBD4" font-size="18">receiver geometry · robust availability</text>\n <path d="M75 395H653" stroke="#497A88" stroke-width="1" opacity=".55"/>\n <text x="75" y="430" fill="#F2F7F8" font-size="15" font-weight="700">Nishat Yasmin Anika</text>\n <text x="75" y="456" fill="#89BAC4" font-size="13" letter-spacing="1.4">FSO  /  PYTHON  /  REPRODUCIBLE METHODS</text>\n</g>\n<g>\n <circle cx="1073" cy="257" r="247" fill="url(#halo)"/>\n <path d="M784 398H1395" stroke="#92CBB3" stroke-width="1.5" opacity=".5"/>\n <path d="M784 392Q907 366 1044 389T1395 379V505H784Z" fill="#163B48" opacity=".75"/>\n <rect x="788" y="332" width="600" height="63" fill="#F5CC88" opacity=".055"/>\n <rect x="788" y="266" width="600" height="66" fill="#91A4B1" opacity=".055"/>\n <rect x="788" y="199" width="600" height="67" fill="#B7DAE6" opacity=".065"/>\n <rect x="788" y="132" width="600" height="67" fill="#73DFE3" opacity=".045"/>\n <g font-family="Arial, Helvetica, sans-serif" font-size="11" fill="#81AEBD" letter-spacing="1.6">\n <text x="1310" y="152">UPPER AIR</text><text x="1290" y="219">CLOUD LAYER</text><text x="1306" y="284">CLEAR AIR</text><text x="1319" y="354">FOG</text>\n </g>\n <path d="M944 406L1136 121" stroke="#61EFEB" stroke-width="16" opacity=".13" filter="url(#soft)"/>\n <path d="M944 406L1136 121" stroke="url(#beam)" stroke-width="3.5" stroke-linecap="round" stroke-dasharray="9 6"/>\n <path d="M944 406L1136 121" stroke="#FCE6B2" stroke-width=".75" opacity=".8"/>\n <g transform="translate(1136 120)">\n  <circle r="41" fill="none" stroke="#6DD9DE" stroke-opacity=".26" stroke-width="1.5"/>\n  <circle r="26" fill="none" stroke="#6DD9DE" stroke-opacity=".42" stroke-width="1.2" stroke-dasharray="3 4"/>\n  <path d="M-38 0H38 M0 -36V36" stroke="#A7DFE0" stroke-width="1.2" opacity=".45"/>\n  <path d="M-32 -2H32 M-15 -12L15 12 M-15 12L15 -12" stroke="#DAFAF6" stroke-width="2"/>\n  <rect x="-13" y="-9" width="26" height="18" rx="4" fill="#71D7D9" stroke="#D2FAF6" stroke-width="1.5"/>\n  <circle r="4" fill="#0A2A3F"/>\n </g>\n <g transform="translate(944 407)" stroke="#C0E5E9" fill="none" stroke-width="2.4">\n  <path d="M-30 0L0 -45L30 0M-20 0H20 M-14 -11H14"/>\n  <path d="M-15 -26L0 -33L14 -26" stroke="#E8CE90"/>\n  <rect x="-38" y="0" width="76" height="9" rx="2" fill="#335D69" stroke="#A2DBD1"/>\n </g>\n <g fill="#B5E8E6" opacity=".65"><circle cx="1045" cy="257" r="2.5"/><circle cx="1090" cy="190" r="2.5"/><circle cx="987" cy="342" r="2.5"/></g>\n <g font-family="Arial, Helvetica, sans-serif" font-size="12" fill="#C5E8E9"><text x="1162" y="95">AERIAL PLATFORM</text><text x="992" y="445">GROUND TERMINAL</text></g>\n <g transform="translate(1190 403)" fill="none" stroke="#77E2DC" stroke-opacity=".42">\n  <circle r="58"/><circle r="37"/><circle r="17"/><path d="M-70 0H70M0 -70V70"/>\n </g>\n</g>\n</svg>', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `assets/method.svg`

In [ ]:
p = PROJECT_ROOT / 'assets/method.svg'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1200 245" role="img" aria-labelledby="t d">\n<title id="t">Research pipeline</title><desc id="d">Four stages: altitude-aware weather model, Gaussian aperture capture, exact optical-power availability, and uncertainty-aware receiver sizing.</desc>\n<rect width="1200" height="245" rx="19" fill="#F3F7FA"/>\n<g font-family="Arial, Helvetica, sans-serif">\n<g transform="translate(24 22)"><rect width="263" height="199" rx="16" fill="#FFF" stroke="#D6E4EB"/><rect x="17" y="17" width="38" height="38" rx="12" fill="#DEF3EF"/><path d="M26 45V31H46M26 38H46M26 31H46" stroke="#198780" stroke-width="2" fill="none"/><text x="69" y="44" fill="#14435C" font-size="13" font-weight="700">01  ATMOSPHERE</text><text x="17" y="95" fill="#142B3A" font-size="19" font-weight="700">Altitude layers</text><text x="17" y="121" fill="#5D7380" font-size="13">Five illustrative weather</text><text x="17" y="140" fill="#5D7380" font-size="13">scenarios; two coefficient</text><text x="17" y="159" fill="#5D7380" font-size="13">interpretations</text><text x="17" y="188" fill="#16867F" font-size="15">T (weather, altitude)</text></g>\n<g transform="translate(319 22)"><rect width="263" height="199" rx="16" fill="#FFF" stroke="#D6E4EB"/><rect x="17" y="17" width="38" height="38" rx="12" fill="#E9F2FF"/><circle cx="36" cy="36" r="11" stroke="#3F7FB1" stroke-width="2" fill="none"/><circle cx="40" cy="33" r="3" fill="#3F7FB1"/><text x="69" y="44" fill="#14435C" font-size="13" font-weight="700">02  GEOMETRY</text><text x="17" y="95" fill="#142B3A" font-size="19" font-weight="700">Beam collection</text><text x="17" y="121" fill="#5D7380" font-size="13">Gaussian spot, circular</text><text x="17" y="140" fill="#5D7380" font-size="13">aperture, stochastic</text><text x="17" y="159" fill="#5D7380" font-size="13">pointing displacement</text><text x="17" y="188" fill="#3F7FB1" font-size="15">G (aperture, jitter)</text></g>\n<g transform="translate(614 22)"><rect width="263" height="199" rx="16" fill="#FFF" stroke="#D6E4EB"/><rect x="17" y="17" width="38" height="38" rx="12" fill="#FAEDE1"/><path d="M25 45L32 37L38 40L46 28" stroke="#C7893C" stroke-width="2.8" fill="none"/><text x="69" y="44" fill="#14435C" font-size="13" font-weight="700">03  PROBABILITY</text><text x="17" y="95" fill="#142B3A" font-size="19" font-weight="700">Link availability</text><text x="17" y="121" fill="#5D7380" font-size="13">Exact finite-state</text><text x="17" y="140" fill="#5D7380" font-size="13">probabilities, checked</text><text x="17" y="159" fill="#5D7380" font-size="13">against simulation</text><text x="17" y="188" fill="#C7893C" font-size="15">P (power ≥ threshold)</text></g>\n<g transform="translate(909 22)"><rect width="267" height="199" rx="16" fill="#FFF" stroke="#D6E4EB"/><rect x="17" y="17" width="38" height="38" rx="12" fill="#EEEAFE"/><path d="M26 43V32L36 26L46 32V43L36 49Z" stroke="#8164BA" stroke-width="2" fill="none"/><text x="69" y="44" fill="#14435C" font-size="13" font-weight="700">04  DESIGN</text><text x="17" y="95" fill="#142B3A" font-size="19" font-weight="700">Robust aperture</text><text x="17" y="121" fill="#5D7380" font-size="13">Minimum receiver size</text><text x="17" y="140" fill="#5D7380" font-size="13">under weather-probability</text><text x="17" y="159" fill="#5D7380" font-size="13">uncertainty</text><text x="17" y="188" fill="#8164BA" font-size="15">min diameter | target</text></g>\n</g></svg>', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

### `.gitignore`

In [ ]:
p = PROJECT_ROOT / '.gitignore'
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('__pycache__/\n*.pyc\n.ipynb_checkpoints/\n.DS_Store\n', encoding='utf-8')
print('Wrote', p.relative_to(PROJECT_ROOT))

## 3. Run numerical regression tests
The run stops if any test fails. These tests verify the model implementation, not its empirical accuracy.

In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=PROJECT_ROOT,check=True)

## 4. Run full experiments
Full mode uses 100,000 samples per largest MC run and each multi-case run. Set `FAST_MODE=True` for 20,000 samples. Analytic results are unchanged.

In [ ]:
import os
FAST_MODE = False
subprocess.run([sys.executable,'run_study.py'],cwd=PROJECT_ROOT,env={**os.environ,'FAST_MODE':'1' if FAST_MODE else '0'},check=True)
subprocess.run([sys.executable,'verify_manifest.py'],cwd=PROJECT_ROOT,check=True)

## 5. Inspect results
The three figures below compare both source conventions, atmospheric models and fog coefficient assumptions. All twelve figures and eleven CSV tables are exported.

In [ ]:
from IPython.display import display, Image
import csv
for name in ('10_optimization_convention_sensitivity','11_model_and_target_sensitivity','12_fog_model_benchmark'):
    display(Image(filename=str(PROJECT_ROOT/'results'/'figures'/(name+'.png'))))
with (PROJECT_ROOT/'results/tables/09_model_and_target_sensitivity.csv').open() as f:
    for row in csv.DictReader(f):
        if float(row['target']) == .92:
            print(row['profile'], 'TV='+row['tv_radius'], 'diameter_cm='+row['diameter_cm'], 'ceiling='+row['finite_aperture_ceiling'], row['status'])

## 6. Export GitHub-ready update
Download `github_update.zip` from Colab Files. This archive uses repository-relative paths. Preserve the existing `.git` history. License, citation metadata and assets are included. Review `UPDATE_GITHUB.md` before uploading.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
subprocess.run([sys.executable,'build_notebook.py'],cwd=PROJECT_ROOT,check=True)
archive=Path.cwd()/'github_update.zip'
with ZipFile(archive,'w',ZIP_DEFLATED,compresslevel=8) as z:
    for p in sorted(PROJECT_ROOT.rglob('*')):
        if p.is_file() and '__pycache__' not in p.parts and not p.name.endswith('.pyc'):
            z.write(p,p.relative_to(PROJECT_ROOT).as_posix())
print('Ready:',archive)
# Optional in Google Colab:
# from google.colab import files; files.download(str(archive))